In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In this notebook I used time-resolved mixed-effects regression to identify potential "biomarkers" - patterns of HFA that:

1. Change consistently wrt behavior across electrodes, across subjects, while accounting for inter-electrode differences in intercept
2. Survive permutation correction (trial-shifting)
3. Survive multiple-comparison correction across time (FDR-BY)

I then run single-electrode, permutation-corrected regressions and characterize the "biomarker" rank of each electrode, both within subjects and across all electrodes. 

In [ ]:
import numpy as np
import mne
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import pandas as pd
from statsmodels.stats.multitest import multipletests
from mne.stats import permutation_cluster_1samp_test
from scipy.signal import filtfilt


# Import plotting functions
# Import the FOOOF object
from tqdm import tqdm
import os
import warnings 

# I only want to see warnings once
warnings.filterwarnings('ignore')

plot_dir = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-TimeResolvedRegression_5_15'
os.makedirs(plot_dir, exist_ok=True)

Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')


In [ ]:
from LFPAnalysis import statistics_utils


In [ ]:
from plot_config import set_plot_style
set_plot_style()

## First, get the behavior data

This is from another notebook with a kernel and environment specifically for beahvior

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site!='UI']
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)



combined_df['gamble_rt_log'] = np.log(combined_df.gamble_rt_gamble)
combined_df['recog_rt_log'] = np.log(combined_df.recog_rt)

combined_df['memory_outcome'] = combined_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome') 

In [ ]:
# # First, let's drop reaction times that are physiologically too fast
# learn_df = learn_df[learn_df.gamble_rt>0.3]
# drop_trials = pd.concat([combined_df[combined_df.gamble_rt_gamble<0.3], 
#                           combined_df[combined_df.recog_rt<0.3]])
# combined_df.drop(index=drop_trials.index, inplace=True)

# # Next, let's drop reaction times that are slower than the 97.5th percentile of each subject's reaction times
# for subj in learn_df.participant.unique():
#     rt_threshold = np.percentile(learn_df[learn_df.participant==subj].gamble_rt, 95)
#     learn_df.loc[(learn_df.participant==subj) & (learn_df.gamble_rt>rt_threshold), 'gamble_rt'] = np.nan
#     combined_df.loc[(combined_df.participant==subj) & (combined_df.gamble_rt_gamble>rt_threshold), 'gamble_rt_gamble'] = np.nan

# learn_df.dropna(subset=['gamble_rt'], inplace=True)
# combined_df.dropna(subset=['gamble_rt_gamble'], inplace=True)

# 3/18/2025

# recover the latent Q-values
combined_df['Q'] = combined_df.rpe + combined_df.reward_gamble
# compute the prior trial's rpe:
combined_df['prior_rpe'] = combined_df.groupby('participant')['rpe'].shift(1)
# compute the trials since last reversal
reversal_trials = [16, 33, 48, 65]
combined_df['trials_since_reversal'] = combined_df.apply(lambda x: np.mod(x.trials_gamble, reversal_trials[int(x.block_gamble-2)]) if x.block_gamble>1 else x.trials_gamble, axis=1)
# recode hits as 0.5 and misses as -0.5
combined_df['phit'] = combined_df.apply(lambda x: 0.5 if x.hits==True else -0.5 if x.hits==False else np.nan, axis=1)
# recode reward  as 0.5 and non-reward as -0.5
combined_df['preward'] = combined_df.apply(lambda x: 0.5 if x.reward_gamble==1 else -0.5 if x.reward_gamble==0 else np.nan, axis=1)
# recode choice==1 as 0.5 and choice==0 as -0.5
combined_df['pgamble'] = combined_df.apply(lambda x: 0.5 if x.gamble_gamble==1 else -0.5 if x.gamble_gamble==0 else np.nan, axis=1)

# Gelman standardization 
for subj in combined_df.participant.unique():

        combined_df.loc[combined_df.participant==subj, 
                'zQ'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'Q'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'Q'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'Q']))
        combined_df.loc[combined_df.participant==subj,
                'zprior_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'prior_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'prior_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'prior_rpe']))
        combined_df.loc[combined_df.participant==subj, 
                        'zrpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                                'rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                        'rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                        'rpe']))
        combined_df.loc[combined_df.participant==subj, 
                'zpm'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'DPRIME'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'DPRIME'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'DPRIME']))

        combined_df.loc[combined_df.participant==subj,
                'ztrials_since_reversal'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'trials_since_reversal'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'trials_since_reversal'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'trials_since_reversal']))
        # compute the reaction-time based EV with res[ect to each subject's min and max RT
        RT_inferred_EV_subj =  1 - ((combined_df.loc[combined_df.participant==subj, 
                                                     'gamble_rt_log'] - np.nanmin(combined_df.loc[combined_df.participant==subj, 
                                                                                                  'gamble_rt_log'])) / (np.nanmax(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                                  'gamble_rt_log']) - np.nanmin(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                                                                                'gamble_rt_log'])))
        # compute 
        combined_df.loc[combined_df.participant==subj,
                'RT_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'reward_gamble'] - RT_inferred_EV_subj)

        combined_df.loc[combined_df.participant==subj,
                'zRT_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'RT_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'RT_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'RT_rpe']))


In [ ]:
# 3/18/2025

# recover the latent Q-values
learn_df['Q'] = learn_df.rpe + learn_df.reward
# compute the prior trial's rpe:
learn_df['prior_rpe'] = learn_df.groupby('participant')['rpe'].shift(1)
# compute the trials since last reversal
reversal_trials = [16, 33, 48, 65]
learn_df['trials_since_reversal'] = learn_df.apply(lambda x: np.mod(x.trials, reversal_trials[int(x.block-2)]) if x.block>1 else x.trials, axis=1)
# recode reward  as 0.5 and non-reward as -0.5
learn_df['preward'] = learn_df.apply(lambda x: 0.5 if x.reward==1 else -0.5 if x.reward==0 else np.nan, axis=1)
# recode choice==1 as 0.5 and choice==0 as -0.5
learn_df['pgamble'] = learn_df.apply(lambda x: 0.5 if x.gamble==1 else -0.5 if x.gamble==0 else np.nan, axis=1)

# Gelman standardization 
for subj in learn_df.participant.unique():

        learn_df.loc[learn_df.participant==subj, 
                'zQ'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'Q'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'Q'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'Q']))
        learn_df.loc[learn_df.participant==subj,
                'zprior_rpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'prior_rpe'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'prior_rpe'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'prior_rpe']))
        learn_df.loc[learn_df.participant==subj, 
                        'zrpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                                'rpe'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                        'rpe'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                        'rpe']))
        learn_df.loc[learn_df.participant==subj, 
                'zpm'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'DPRIME'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'DPRIME'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'DPRIME']))

        learn_df.loc[learn_df.participant==subj,
                'ztrials_since_reversal'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'trials_since_reversal'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'trials_since_reversal'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'trials_since_reversal']))
        # compute the reaction-time based EV with res[ect to each subject's min and max RT
        RT_inferred_EV_subj =  1 - ((learn_df.loc[learn_df.participant==subj, 
                                                     'gamble_rt_log'] - np.nanmin(learn_df.loc[learn_df.participant==subj, 
                                                                                                  'gamble_rt_log'])) / (np.nanmax(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                                  'gamble_rt_log']) - np.nanmin(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                                                                                'gamble_rt_log'])))
        # compute 
        learn_df.loc[learn_df.participant==subj,
                'RT_rpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'reward'] - RT_inferred_EV_subj)

        learn_df.loc[learn_df.participant==subj,
                'zRT_rpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'RT_rpe'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'RT_rpe'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'RT_rpe']))
        # tercile split the rpe into rpe_categories: 'negative', 'neutral', 'positive' using pd.qcut
        learn_df.loc[learn_df.participant==subj,
                'rpe_cat'] = pd.qcut(learn_df.loc[learn_df.participant==subj, 
                                                                        'rpe'], 
                                                                        q=3, 
                                                                        labels=['negative', 'neutral', 'positive'])





# 1/9/2025: Mixed-effects regression

I want to do a mixed-effects model at each timestamp. This means somehow loading each timestamp separately and making a dataframe for it. 

In [ ]:
# progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
# freqs = np.logspace(*np.log10([2, 200]), num=30)
# event = 'feedback_start'

# freq_dict = {'theta':[2, 8], 
#             'alpha':[8, 13],
#             'beta': [13, 30], 
#             'hfa': [70, 200]}

# rois = ['OFC']
# day = 'Day1'
# y='tfr'
# formula='tfr ~ 1 + hemi + zpm*phit + zrpe*phit'   
# win_len = 75 # samples 
# slide_len = 50 # samples 
# sr=500
# nsurr = 100

# combined_df = combined_df[combined_df.condition=='Day1']

# for band in ['hfa']:
#     for region in rois:
#         model_dfs = []
#         for subj in subj_ids: 
#             combined_df.loc[combined_df.participant==subj, 
#                             'zrpe'] = (combined_df.loc[combined_df.participant==subj, 
#                                                                                     'rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
#                                                                                                             'rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
#                                                                                                                                                 'rpe']))
#             combined_df.loc[combined_df.participant==subj, 
#                     'zpm'] = (combined_df.loc[combined_df.participant==subj, 
#                                                                             'DPRIME'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
#                                                                                                     'DPRIME'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
#                                                                                                                                         'DPRIME']))            
#             combined_df['phit'] = combined_df.apply(lambda x: 0.5 if x.hits==True else -0.5, axis=1)
#             combined_df['reward'] = combined_df.rpe.apply(lambda x: 0.5 if x > 0 else -0.5)

#             if event == 'cue_on':
#                 trial_var = 'trials_mem'
#             else:
#                 trial_var = 'trials_gamble'

#             if event == 'cue_on':
#                 regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'zrpe', 'zpm', 'phit', f'{trial_var}', 'reward', 'recog_rt']]
#             else:
#                 regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'zrpe', 'zpm', 'phit', f'{trial_var}', 'reward']]

#             elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/{day}_reref_elec_df')
#             filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/scratch/TFR'
#             tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')

#             # get all the labels in elec_df for which salman region is in rois
#             roi_index = elec_df.loc[elec_df.salman_region==region, 'label'].values.tolist()

#             if len(roi_index)==0:
#                 continue

#             # get the average time series across frequenceis for all channels in the roi
#             timeseries = np.nanmean(tfr_data.copy().pick(roi_index).data[:, :,
#                                                                         np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
#                                                                         :], axis=2)

#             # define slices for smoothing timeseries: 
#             slices = np.lib.stride_tricks.sliding_window_view(np.arange(tfr_data.data.shape[-1]), win_len)[::slide_len]
#             midpoints = np.ceil(np.mean(slices, axis=1))
#             ts = midpoints * (1000/sr)

#             sig = np.zeros([timeseries.shape[0], timeseries.shape[1], slices.shape[0]])
#             for stride in range(slices.shape[0]):
#                 sig[:, :, stride] = np.nanmean(timeseries[:, :, slices[stride]], axis=-1)

#             for j, ch in enumerate(roi_index):
#                     for i in range(len(ts)):
#                         ch_df = regressors.copy()
#                         ch_df['ts'] = ts[i]
#                         ch_df.rename(columns={f'{trial_var}':'trial'}, inplace=True)
#                         # ch_df.dropna()
#                         # sort by whatever trials we are using
#                         ch_df.sort_values(by=['trial'], inplace=True)
#                         trial_ix = ch_df.trial.values.astype(int) - 1
#                         ch_df['tfr'] = sig[trial_ix, j, i]
#                         ch_df['unique_label'] = f'{subj}_{ch}'
#                         ch_df['hemi'] = elec_df.loc[elec_df.label==ch, 'hemisphere'].str.lower().values[0]
                        
#                         model_dfs.append(ch_df)

#         full_df = pd.concat(model_dfs).dropna(subset=['zrpe'])
#         if event == 'cue_on':
#             full_df = full_df[full_df.recog_rt>=1]
#         full_df['band'] = band
#         full_df['region'] = region

#         full_df.to_csv(f'{load_dir}/full_df_time_resolved_mlm_{event}_{band}_{region}.csv', index=False)

#         # # cap out the ludicrous z-scores which are also partially mitigated by permutations
#         # full_df.loc[full_df.tfr>10, 'tfr'] = 10

#         all_res = statistics_utils.time_resolved_mlm(full_df,
#                             y=y,
#                             formula=formula,
#                             lower_group='unique_label',
#                             higher_group='participant',
#                             trial_key='trial',
#                             n_permutations=nsurr,
#                             n_jobs=-1)
 
#         all_res.to_csv(f'{load_dir}/all_res_time_resolved_mlm_{event}_{band}_{region}.csv', index=False)


In [ ]:
# event = 'feedback_start'
# band = 'hfa'
# region = 'AMY'
# full_df = pd.read_csv(f'{load_dir}/full_df_time_resolved_mlm_{event}_{band}_{region}.csv')
# OFC_power = full_df[(full_df.ts>=300) & (full_df.ts<=550)& (full_df.tfr<6)].groupby(['participant', 'unique_label', 'trial']).mean(numeric_only=True).reset_index()


In [ ]:
# event = 'feedback_start'
# band = 'hfa'
# region = 'OFC'
# full_df = pd.read_csv(f'{load_dir}/full_df_time_resolved_mlm_{event}_{band}_{region}.csv')
# OFC_power = full_df[(full_df.ts>=300) & (full_df.ts<=550)& (full_df.tfr<10)].groupby(['participant', 'unique_label', 'trial']).mean(numeric_only=True).reset_index()

# event = 'feedback_start'
# region = 'HPC'
# full_df = pd.read_csv(f'{load_dir}/full_df_time_resolved_mlm_{event}_{band}_{region}.csv')
# HPC_power = full_df[(full_df.ts>=800) & (full_df.ts<=1000) & (full_df.tfr<10)].groupby(['participant', 'unique_label', 'trial']).mean(numeric_only=True).reset_index()

# event = 'feedback_start'
# region = 'AMY'
# full_df = pd.read_csv(f'{load_dir}/full_df_time_resolved_mlm_{event}_{band}_{region}.csv')
# AMY_power = full_df[(full_df.ts>=300) & (full_df.ts<=450) & (full_df.tfr<10)].groupby(['participant', 'unique_label', 'trial']).mean(numeric_only=True).reset_index()


# event = 'feedback_start'
# region = 'dmPFC'
# full_df = pd.read_csv(f'{load_dir}/full_df_time_resolved_mlm_{event}_{band}_{region}.csv')
# dmPFC_power = full_df[(full_df.ts>=1250) & (full_df.ts<=1500) & (full_df.tfr<10)].groupby(['participant', 'unique_label', 'trial']).mean(numeric_only=True).reset_index()


In [ ]:
# event = 'feedback_start'
# band = 'hfa'
# region = 'OFC'
# full_df = pd.read_csv(f'{load_dir}/full_df_time_resolved_mlm_{event}_{band}_{region}.csv')
# # OFC_power = full_df[(full_df.ts>=300) & (full_df.ts<=550)& (full_df.tfr<6)].groupby(['participant', 'unique_label', 'trial']).mean(numeric_only=True).reset_index()


# OFC_power1 = full_df[(full_df.ts>=300) & (full_df.ts<=550)& (full_df.tfr<10)].groupby(['participant', 'unique_label', 'trial']).mean(numeric_only=True).reset_index()
# OFC_power2 = full_df[(full_df.ts>=1250) & (full_df.ts<=1500)& (full_df.tfr<10)].groupby(['participant', 'unique_label', 'trial']).mean(numeric_only=True).reset_index()
# merged_OFC_df = pd.merge(OFC_power1, OFC_power2, on=['participant', 'unique_label', 'trial'], suffixes=('_1', '_2'))

## 1/23/25: Enable multinomial memory regression for cue period

In [ ]:
# import statsmodels as sm

# def multinomial_regression(df, response_col, predictor_col):
#     # Drop rows with NaN values in the dependent or predictor columns
#     rel_cols = [response_col] + predictor_col
#     df_clean = df[rel_cols].dropna()

#     # Encode the categorical response variable
#     # label_encoder = LabelEncoder()
#     # df_clean['response_encoded'] = label_encoder.fit_transform(df_clean[response_col].values.ravel())
#     df_clean['response_encoded'] = df_clean[response_col].apply(lambda x: 0 if x=='hits' else 1 if x=='misses' else 2 if x=='false_alarms' else 3).values.ravel()
    
#     # Prepare the predictor variable (add constant for intercept)
#     X = sm.add_constant(df_clean[predictor_col])
    
#     # The dependent variable is the encoded response
#     y = df_clean['response_encoded']
        
#     # Fit the multinomial logistic regression model
#     model = sm.MNLogit(y, X)
#     result = model.fit(disp=0)
    
#     # Return the model result and the label encoder for further use
#     return result

# # Example usage
# # result = multinomial_regression(combined_df, 'memory_outcome', ['recog_rt_log'])
# # any(result.pvalues.loc['recog_rt_log'].values <=0.05)

In [ ]:
# progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
# freqs = np.logspace(*np.log10([2, 200]), num=30)
# event = 'cue_on'

# freq_dict = {'theta':[2, 8], 
#             'alpha':[8, 13],
#             'beta': [13, 30], 
#             'hfa': [70, 200]}

# rois = ['OFC', 'dmPFC', 'HPC', 'ACC', 'AMY', 'dlPFC'] # 'OFC'
# day = 'Day1'
# y='tfr'
# formula='tfr ~ 1 + hemi + phit + log_recog_rt'   
# win_len = 50 # samples 
# slide_len = 25 # samples 
# sr=500
# nsurr = 100

# combined_df = combined_df[combined_df.condition=='Day1']

# for band in ['hfa']:
#     for region in rois:
#         model_dfs = []
#         for subj in subj_ids: 
#             combined_df.loc[combined_df.participant==subj, 
#                             'zrpe'] = (combined_df.loc[combined_df.participant==subj, 
#                                                                                     'rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
#                                                                                                             'rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
#                                                                                                                                                 'rpe']))
#             combined_df.loc[combined_df.participant==subj, 
#                     'zpm'] = (combined_df.loc[combined_df.participant==subj, 
#                                                                             'DPRIME'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
#                                                                                                     'DPRIME'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
#                                                                                                                                         'DPRIME']))            
#             combined_df['phit'] = combined_df.apply(lambda x: 0.5 if x.hits==True else -0.5, axis=1)
#             combined_df['reward'] = combined_df.rpe.apply(lambda x: 0.5 if x > 0 else -0.5)

#             if event == 'cue_on':
#                 trial_var = 'trials_mem'
#             else:
#                 trial_var = 'trials_gamble'

#             if event == 'cue_on':
#                 regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'zrpe', 'zpm', 'phit', f'{trial_var}', 'reward', 'recog_rt']]
#             else:
#                 regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'zrpe', 'zpm', 'phit', f'{trial_var}', 'reward']]

#             elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/{day}_reref_elec_df')
#             filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/scratch/TFR'
#             tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')

#             # get all the labels in elec_df for which salman region is in rois
#             roi_index = elec_df.loc[elec_df.salman_region==region, 'label'].values.tolist()

#             if len(roi_index)==0:
#                 continue

#             # get the average time series across frequenceis for all channels in the roi
#             timeseries = np.nanmean(tfr_data.copy().pick(roi_index).data[:, :,
#                                                                         np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
#                                                                         :], axis=2)

#             # define slices for smoothing timeseries: 
#             slices = np.lib.stride_tricks.sliding_window_view(np.arange(tfr_data.data.shape[-1]), win_len)[::slide_len]
#             midpoints = np.ceil(np.mean(slices, axis=1))
#             ts = midpoints * (1000/sr)

#             sig = np.zeros([timeseries.shape[0], timeseries.shape[1], slices.shape[0]])
#             for stride in range(slices.shape[0]):
#                 sig[:, :, stride] = np.nanmean(timeseries[:, :, slices[stride]], axis=-1)

#             for j, ch in enumerate(roi_index):
#                     for i in range(len(ts)):
#                         ch_df = regressors.copy()
#                         ch_df['ts'] = ts[i]
#                         ch_df.rename(columns={f'{trial_var}':'trial'}, inplace=True)
#                         # ch_df.dropna()
#                         # sort by whatever trials we are using
#                         ch_df.sort_values(by=['trial'], inplace=True)
#                         trial_ix = ch_df.trial.values.astype(int) - 1
#                         ch_df['tfr'] = sig[trial_ix, j, i]
#                         ch_df['unique_label'] = f'{subj}_{ch}'
#                         ch_df['hemi'] = elec_df.loc[elec_df.label==ch, 'hemisphere'].str.lower().values[0]
                        
#                         model_dfs.append(ch_df)

#         full_df = pd.concat(model_dfs).dropna(subset=['zrpe'])
#         if event == 'cue_on':
#             full_df = full_df[full_df.recog_rt>=1]
#         full_df['band'] = band
#         full_df['region'] = region


#         full_df['log_recog_rt'] = np.log10(full_df.recog_rt)
#         full_df.to_csv(f'{load_dir}/full_df_time_resolved_mlm_{event}_{band}_{region}.csv', index=False)

#         # # cap out the ludicrous z-scores which are also partially mitigated by permutations
#         # full_df.loc[full_df.tfr>10, 'tfr'] = 10
#         # full_df.loc[full_df.tfr<-10, 'tfr'] = -10

#         all_res = statistics_utils.time_resolved_mlm(full_df,
#                             y=y,
#                             formula=formula,
#                             lower_group='unique_label',
#                             higher_group='participant',
#                             trial_key='trial',
#                             n_permutations=nsurr,
#                             n_jobs=-1)
 
#         all_res.to_csv(f'{load_dir}/all_res_time_resolved_mlm_{event}_{band}_{region}.csv', index=False)


## Plotting encoding results

## Plotting cue on results

In [ ]:
# band = 'hfa'
# win_len = 50
# sr = 500
# slide_len = 25  
# # correct multiple comparisons across timepoints and regions 

# event = 'cue_on'

# all_res_dfs = []
# for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
#     all_res = pd.read_csv(f'{load_dir}/all_res_time_resolved_mlm_{event}_{band}_{region}.csv')
#     all_res['mc_p'] = np.nan
#     all_res['region'] = region
#     all_res.count_p = all_res.count_p/2
#     all_res.loc[all_res.parameter=='phit', 'mc_p'] = multipletests(all_res.loc[all_res.parameter=='phit', 'raw_p'], method='fdr_bh')[1]

#     fig, ax = plt.subplots(1, 1, figsize=(6, 5))
#     if region=='OFC':
#         color = ofc_c
#     elif region=='HPC': 
#         color = hpc_c
#     elif region=='AMY':
#         color = amy_c
#     elif region=='dmPFC':
#         color = dmpfc_c
#     elif region=='dlPFC':
#         color = dlpfc_c
#     elif region=='ACC':
#         color = acc_c

#     # Add horizontal lines
#     line_width = win_len * (1000/sr)

#     # plot bars centered at 'ts', with height 'z_beta', and width 'line_width'. make them gray, unless mc_p<=0.05, in which case color=color
#     plt.bar(all_res[(all_res.parameter=='phit') & (all_res.mc_p>0.05)]['ts'],
#             all_res[(all_res.parameter=='phit') & (all_res.mc_p>0.05)]['z_beta'],
#             width=line_width, color='black', alpha=0.2, edgecolor='black', linewidth=1)
#     plt.bar(all_res[(all_res.parameter=='phit') & (all_res.mc_p<=0.05)]['ts'],
#             all_res[(all_res.parameter=='phit') & (all_res.mc_p<=0.05)]['z_beta'],
#             width=line_width, color=color, alpha=1, linewidth=1)
#     plt.plot(all_res[(all_res.parameter=='phit')]['ts'],
#         all_res[(all_res.parameter=='phit')]['z_beta'],
#         color='black', alpha=1, linewidth=2)
    
#     plt.hlines(0, 0, 1000, linestyle='--', color='black')
#     sns.despine()
#     ax.set(title=f'{region}', ylabel='rpe:memory interaction (z)', xlabel='time-bin (ms)')

#     plt.savefig(f'/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots/time_resolved_mlm_{event}_{band}_{region}.pdf', dpi=300)

In [ ]:
# band = 'hfa'
# win_len = 50
# sr = 500
# slide_len = 25  
# # correct multiple comparisons across timepoints and regions 

# event = 'cue_on'

# all_res_dfs = []
# for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
#     all_res = pd.read_csv(f'{load_dir}/all_res_time_resolved_mlm_{event}_{band}_{region}.csv')
#     all_res['mc_p'] = np.nan
#     all_res['region'] = region
#     all_res.count_p = all_res.count_p/2
#     all_res.loc[all_res.parameter=='zrpe:phit', 'mc_p'] = multipletests(all_res.loc[all_res.parameter=='zrpe:phit', 'raw_p'], method='fdr_bh')[1]

#     fig, ax = plt.subplots(1, 1, figsize=(6, 5))
#     if region=='OFC':
#         color = ofc_c
#     elif region=='HPC': 
#         color = hpc_c
#     elif region=='AMY':
#         color = amy_c
#     elif region=='dmPFC':
#         color = dmpfc_c
#     elif region=='dlPFC':
#         color = dlpfc_c
#     elif region=='ACC':
#         color = acc_c

#     # Add horizontal lines
#     line_width = win_len * (1000/sr)

#     # plot bars centered at 'ts', with height 'z_beta', and width 'line_width'. make them gray, unless mc_p<=0.05, in which case color=color
#     plt.bar(all_res[(all_res.parameter=='zrpe:phit') & (all_res.mc_p>0.05)]['ts'],
#             all_res[(all_res.parameter=='zrpe:phit') & (all_res.mc_p>0.05)]['z_beta'],
#             width=line_width, color='black', alpha=0.2, edgecolor='black', linewidth=1)
#     plt.bar(all_res[(all_res.parameter=='zrpe:phit') & (all_res.mc_p<=0.05)]['ts'],
#             all_res[(all_res.parameter=='zrpe:phit') & (all_res.mc_p<=0.05)]['z_beta'],
#             width=line_width, color=color, alpha=1, linewidth=1)
#     plt.plot(all_res[(all_res.parameter=='zrpe:phit')]['ts'],
#         all_res[(all_res.parameter=='zrpe:phit')]['z_beta'],
#         color='black', alpha=1, linewidth=2)
    
#     plt.hlines(0, 0, 1000, linestyle='--', color='black')
#     sns.despine()
#     ax.set(title=f'{region}', ylabel='rpe:memory interaction (z)', xlabel='time-bin (ms)')

#     plt.savefig(f'/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots/time_resolved_mlm_{event}_{band}_{region}.pdf', dpi=300)

Note (1/22/2025): These results come after a) altering the baseline approach, and b) removing all artifact/IED filtering. We trust in permutations a lot here

# Single electrode analysis

In the single-electrode framework, I can use the permutation-cluster test to determine significance: 

## Encoding

2/26/2025: I need to look at the choice time. I think it depends on the prior RPE, and the trials since last reversal

3/18/2025: I need to run my time-wise regression with the following: 

1. preward

2. rpe 

3. Q 

4. RT_rpe 

In [ ]:

# # combined_df = combined_df[combined_df.condition=='Day1']
# learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH_ddm.csv')
# # merge the combined_df with learn_df on participant and trials_gamble (called "trials" in learn_df)
# learn_df.rename(columns={'trials':'trials_gamble'}, inplace=True)
# combined_df=combined_df[combined_df.condition=='Day1']
# combined_df = combined_df[['participant', 'trials_gamble', 'zrpe', 'reward_y', 'preward', 'zpm', 'hits', 'zRT_rpe']].merge(learn_df[['participant', 'trials_gamble', 'drift_mean', 'rpe_ddm']], on=['participant', 'trials_gamble'], how='left')

# # Gelman standardize the drift_mean by subject
# for subj in combined_df.participant.unique():
#     combined_df.loc[combined_df.participant==subj, 'zdrift_mean'] = (combined_df.loc[combined_df.participant==subj, 'drift_mean'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 'drift_mean'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 'drift_mean']))
#     combined_df.loc[combined_df.participant==subj, 'zrpe_ddm'] = (combined_df.loc[combined_df.participant==subj, 'rpe_ddm'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 'rpe_ddm'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 'rpe_ddm']))
#     learn_df.loc[learn_df.participant==subj, 'zdrift_mean'] = (learn_df.loc[learn_df.participant==subj, 'drift_mean'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 'drift_mean'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 'drift_mean']))
#     learn_df.loc[learn_df.participant==subj, 'zrpe_ddm'] = (learn_df.loc[learn_df.participant==subj, 'rpe_ddm'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 'rpe_ddm'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 'rpe_ddm']))

# for subj in combined_df.participant.unique():
#     combined_df.loc[combined_df.participant==subj, 'drift_mean_standardized'] = (combined_df.loc[combined_df.participant==subj, 'drift_mean'] - np.nanmin(combined_df.loc[combined_df.participant==subj, 'drift_mean'])) / (np.nanmax(combined_df.loc[combined_df.participant==subj, 'drift_mean']) - np.nanmin(combined_df.loc[combined_df.participant==subj, 'drift_mean']))
#     learn_df.loc[learn_df.participant==subj, 'drift_mean_standardized'] = (learn_df.loc[learn_df.participant==subj, 'drift_mean'] - np.nanmin(learn_df.loc[learn_df.participant==subj, 'drift_mean'])) / (np.nanmax(learn_df.loc[learn_df.participant==subj, 'drift_mean']) - np.nanmin(learn_df.loc[learn_df.participant==subj, 'drift_mean']))

#     # make a new column that is the difference between reward and drift_mean_standardized
# combined_df['reward_minus_drift'] = combined_df['reward_y'] - combined_df['drift_mean_standardized']
# learn_df['reward_minus_drift'] = learn_df['reward'] - learn_df['drift_mean_standardized']


In [ ]:

# combined_df = combined_df[combined_df.condition=='Day1']
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH_ddm_driftRPE.csv')
# merge the combined_df with learn_df on participant and trials_gamble (called "trials" in learn_df)
learn_df.rename(columns={'trials':'trials_gamble'}, inplace=True)
combined_df=combined_df[combined_df.condition=='Day1']
combined_df = combined_df[['participant', 'trials_gamble', 'zrpe', 'reward_y', 'preward', 'zpm', 'hits', 'zRT_rpe']].merge(learn_df[['participant', 'trials_gamble', 'drift_mean', 'rpe_drift']], on=['participant', 'trials_gamble'], how='left')

# Gelman standardize the drift_mean by subject
for subj in combined_df.participant.unique():
    # combined_df.loc[combined_df.participant==subj, 'zdrift_mean'] = (combined_df.loc[combined_df.participant==subj, 'drift_mean'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 'drift_mean'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 'drift_mean']))
    combined_df.loc[combined_df.participant==subj, 'zrpe_drift'] = (combined_df.loc[combined_df.participant==subj, 'rpe_drift'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 'rpe_drift'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 'rpe_drift']))
    learn_df.loc[learn_df.participant==subj, 'zrpe_drift'] = (learn_df.loc[learn_df.participant==subj, 'rpe_drift'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 'rpe_drift'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 'rpe_drift']))




In [ ]:
# progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
# freqs = np.logspace(*np.log10([2, 200]), num=30)
# # event = 'choice_time'

# # event = 'feedback_start'

# freq_dict = {'theta':[2, 8], 
#             'alpha':[8, 13],
#             'beta': [13, 30], 
#             'hfa': [70, 200]}

# rois = ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']
# day = 'Day1'
# y='tfr'
# win_len = 200
# slide_len = 25
# sr=500

# nperm = 100

# for event in ['feedback_start']:
#     for predictor in ['zrpe_drift']:
#         formula=f'tfr ~ 1 + {predictor} + zpm'
#         for band in ['hfa']:
#             model_dfs = []
#             for subj in subj_ids: 
                
#                 if event == 'cue_on':
#                     trial_var = 'trials_mem'
#                 else:
#                     trial_var = 'trials_gamble'

#                 if event == 'cue_on':
#                     regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'phit', f'{trial_var}', f'{predictor}', 'recog_rt_log']]
#                     formula=f'tfr ~ 1 + {predictor}*phit + recog_rt_log'
#                 else:
#                     # regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'phit', f'{trial_var}', f'{predictor}', 'gamble_rt_log']]
#                     regressors = learn_df.loc[learn_df.participant==subj, ['participant', 'zpm', f'{trial_var}', f'{predictor}', 'zrpe']]


#                 elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/{day}_reref_elec_df')
#                 # filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/scratch/TFR'
#                 filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/scratch/TFR'

#                 tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')

#                 # get all the labels in elec_df for which salman region is in rois
#                 roi_index = elec_df.loc[elec_df.salman_region.isin(rois), 'label'].values.tolist()

#                 if len(roi_index)==0:
#                     continue

#                 # get the average time series across frequenceis for all channels in the roi
#                 timeseries = np.nanmean(tfr_data.copy().pick(roi_index).data[:, :,
#                                                                             np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
#                                                                             :], axis=2)
#                 # define slices for smoothing timeseries: 
#                 slices = np.lib.stride_tricks.sliding_window_view(np.arange(tfr_data.data.shape[-1]), win_len)[::slide_len]
#                 midpoints = np.ceil(np.mean(slices, axis=1))
#                 ts = midpoints * (1000/sr)

#                 sig = np.zeros([timeseries.shape[0], timeseries.shape[1], slices.shape[0]])
#                 for stride in range(slices.shape[0]):
#                     sig[:, :, stride] = np.nanmean(timeseries[:, :, slices[stride]], axis=-1)

#                 for j, ch in enumerate(roi_index):
#                     for i in range(len(ts)):
#                         ch_df = regressors.copy()
#                         ch_df['ts'] = ts[i]
#                         ch_df.rename(columns={f'{trial_var}':'trial'}, inplace=True)
#                         ch_df.dropna(subset=[f'{predictor}'], inplace=True)
#                         ch_df.sort_values(by=['trial'], inplace=True)
#                         trial_ix = ch_df.trial.values.astype(int) - 1
#                         ch_df['tfr'] = sig[trial_ix, j, i]
#                         ch_df['unique_label'] = ch
#                         ch_df['region'] = elec_df[elec_df.label==ch].salman_region.values[0] 
#                         ch_df['hemi'] = elec_df.loc[elec_df.label==ch, 'hemisphere'].str.lower().values[0]
#                         # ch_dfs.append(ch_df)
                    
#                         results = statistics_utils.permutation_regression_zscore(ch_df,
#                                 formula=formula,
#                                 n_permutations=nperm, 
#                                 plot_res=False)
#                         results['ts'] = i
#                         results['subj'] = subj
#                         results['unique_label'] = ch
#                         results['region'] = elec_df[elec_df.label==ch].salman_region.values[0]
#                         results['hemi'] = elec_df.loc[elec_df.label==ch, 'hemisphere'].str.lower().values[0]

#                         model_dfs.append(results)
                
#             model_dfs = pd.concat(model_dfs)
#             for ts_i in model_dfs.ts.unique():
#                 model_dfs.loc[model_dfs.ts==ts_i, 'ts'] = midpoints[ts_i] * (1000/sr)
#             model_dfs.to_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_final.csv', index=False)   
        

In [ ]:
f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_final.csv'

In [ ]:
pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_hemi.csv') 

3/18/2025: Make all the time-wise regression plots:

In [ ]:
# Smoothed time-resolved regression (complementary to binning approach above)
# Uses filtfilt smoothing with 200ms window and downsamples by factor of 5
# OPTIMIZED: Parallelized across subjects with checkpointing

from joblib import Parallel, delayed
import os

freqs = np.logspace(*np.log10([2, 200]), num=30)
# event = 'choice_time'

# event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 201]}

rois = ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']
day = 'Day1'
y='tfr'
# Smoothing parameters
win_ms = 200  # 200ms smoothing window
downsample_factor = 5  # Downsample by factor of 5 after smoothing

nperm = 100

# Option to overwrite existing checkpoints (set to True to re-run all subjects)
overwrite_checkpoints = True

# combined_df = combined_df[combined_df.condition=='Day1']

def process_subject_smoothed_regression(subj, event, predictor, band, formula, 
                                        base_dir, load_dir, day, rois, freq_dict, 
                                        freqs, win_ms, downsample_factor, nperm,
                                        learn_df, combined_df):
    """
    Process one subject for smoothed time-resolved regression.
    Returns list of result DataFrames for this subject.
    """
    subject_model_dfs = []
    
    try:
        if event == 'cue_on':
            trial_var = 'trials_mem'
        else:
            trial_var = 'trials_gamble'

        if event == 'cue_on':
            regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'zpm', f'{trial_var}', f'{predictor}']]
        else:
            # regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'phit', f'{trial_var}', f'{predictor}', 'gamble_rt_log']]
            regressors = learn_df.loc[learn_df.participant==subj, ['participant', 'zpm', f'{trial_var}', f'{predictor}']]

        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/{day}_reref_elec_df')
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/scratch/TFR'

        tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')

        # get all the labels in elec_df for which salman region is in rois
        roi_index = elec_df.loc[elec_df.salman_region.isin(rois), 'label'].values.tolist()

        if len(roi_index)==0:
            return subject_model_dfs

        # get the average time series across frequencies for all channels in the roi
        timeseries = np.nanmean(tfr_data.copy().pick(roi_index).data[:, :,
                                                                    np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                                    :], axis=2)
        
        # Apply filtfilt smoothing (200ms window)
        win_size = int(tfr_data.info['sfreq'] * (win_ms / 1000))  # Convert 200ms to samples
        window = np.ones(win_size) / win_size
        
        smoothed_signal = np.zeros_like(timeseries)
        for electrode_idx in range(timeseries.shape[1]):
            smoothed_signal[:, electrode_idx, :] = filtfilt(window, [1], 
                                                           timeseries[:, electrode_idx, :], axis=-1)
        
        # Downsample by factor of 5
        downsampled_signal = smoothed_signal[:, :, ::downsample_factor]
        
        # Get downsampled time vector (convert to ms)
        ts = tfr_data.times[::downsample_factor] * 1000

        for j, ch in enumerate(roi_index):
            for i in range(len(ts)):
                ch_df = regressors.copy()
                ch_df['ts'] = ts[i]
                ch_df.rename(columns={f'{trial_var}':'trial'}, inplace=True)
                ch_df.dropna(subset=[f'{predictor}'], inplace=True)
                ch_df.sort_values(by=['trial'], inplace=True)
                trial_ix = ch_df.trial.values.astype(int) - 1
                # Only take the trials that exist in the behavioral data
                ch_df['tfr'] = downsampled_signal[trial_ix, j, i]
                ch_df['unique_label'] = ch
                ch_df['region'] = elec_df[elec_df.label==ch].salman_region.values[0] 
                ch_df['hemi'] = elec_df.loc[elec_df.label==ch, 'hemisphere'].str.lower().values[0]
            
                results = statistics_utils.permutation_regression_zscore(ch_df,
                        formula=formula,
                        n_permutations=nperm, 
                        plot_res=False)
                results['ts'] = ts[i]  # Store actual time value in ms
                results['subj'] = subj
                results['unique_label'] = ch
                results['region'] = elec_df[elec_df.label==ch].salman_region.values[0]
                results['hemi'] = elec_df.loc[elec_df.label==ch, 'hemisphere'].str.lower().values[0]

                subject_model_dfs.append(results)
        
        # Clear large arrays to free memory
        del timeseries, smoothed_signal, downsampled_signal, tfr_data
        
    except Exception as e:
        print(f"Error processing subject {subj}: {e}")
        return subject_model_dfs
    
    return subject_model_dfs

# Main processing loop with parallelization and checkpointing
for event in ['feedback_start']:
    for predictor in ['zrpe_drift']:
        if predictor == 'rpe_cat':
            formula=f"tfr ~ 1 + C(rpe_cat, Treatment(reference='neutral')) + zpm"
        else:
            formula=f'tfr ~ 1 + {predictor} + zpm'
        for band in ['hfa']:
            # Create checkpoint directory for intermediate saves
            checkpoint_dir = f'{load_dir}/checkpoints_smoothed_{band}_{event}_{predictor}'
            os.makedirs(checkpoint_dir, exist_ok=True)
            
            # Check for existing checkpoints (unless overwrite is enabled)
            completed_subjects = []
            if not overwrite_checkpoints:
                for subj in subj_ids:
                    checkpoint_file = f'{checkpoint_dir}/{subj}_results.csv'
                    if os.path.exists(checkpoint_file):
                        completed_subjects.append(subj)
                        print(f"Found checkpoint for {subj}, skipping...")
            else:
                print("Overwrite mode enabled: will re-process all subjects")
            
            # Get subjects that still need processing
            remaining_subjects = [s for s in subj_ids if s not in completed_subjects]
            
            if len(remaining_subjects) == 0:
                print("All subjects already processed. Loading from checkpoints...")
            else:
                print(f"Processing {len(remaining_subjects)} subjects (skipping {len(completed_subjects)} already completed)...")
                
                # Parallel processing with limited jobs to avoid memory issues
                n_jobs = min(8, os.cpu_count())
                print(f"Using {n_jobs} parallel workers...")
                
                all_results = Parallel(n_jobs=n_jobs, verbose=1)(
                    delayed(process_subject_smoothed_regression)(
                        subj, event, predictor, band, formula,
                        base_dir, load_dir, day, rois, freq_dict,
                        freqs, win_ms, downsample_factor, nperm,
                        learn_df, combined_df
                    )
                    for subj in remaining_subjects
                )
                
                # Save checkpoints for each subject
                print("Saving checkpoints...")
                for subj, subject_results in zip(remaining_subjects, all_results):
                    if len(subject_results) > 0:
                        subject_df = pd.concat(subject_results)
                        checkpoint_file = f'{checkpoint_dir}/{subj}_results.csv'
                        subject_df.to_csv(checkpoint_file, index=False)
                        print(f"  Saved checkpoint for {subj}: {len(subject_results)} results")
            
            # Load all results from checkpoints
            print("Loading all results from checkpoints...")
            all_model_dfs = []
            for subj in subj_ids:
                checkpoint_file = f'{checkpoint_dir}/{subj}_results.csv'
                if os.path.exists(checkpoint_file):
                    subj_df = pd.read_csv(checkpoint_file)
                    all_model_dfs.append(subj_df)
                    print(f"  Loaded {subj}: {len(subj_df)} results")
            
            # Concatenate all results
            if len(all_model_dfs) > 0:
                model_dfs = pd.concat(all_model_dfs, ignore_index=True)
                print(f"Total results: {len(model_dfs)}")
                
                # Save final combined results
                output_file = f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_smoothed_final.csv'
                model_dfs.to_csv(output_file, index=False)
                print(f"Saved final results to: {output_file}")
            else:
                print("No results to save!")
        

In [ ]:
sns.lmplot(data=learn_df, y='rpe_drift', x='reward')
# sns.regplot(data=learn_df, y='rpe', x='reward')

plt.ylabel('zcpe')
# sns.lineplot(x=[-1, 1], y = [-1, 1], color = 'k')

In [ ]:
sns.scatterplot(data=learn_df, x='rpe', y='rpe_drift', hue='reward')
plt.ylabel('zcpe')
sns.lineplot(x=[-1, 1], y = [-1, 1], color = 'k')

In [ ]:
from scipy.stats import pearsonr
pearsonr(learn_df.rpe, learn_df.rpe_drift)

CPE and RPE are highly correlated, though RPE lives a lot near 0, and CPE is very rarely around 0. 

The biggest difference between RPE and CPE is that when RPE is near 0 (happens a lot), CPE can be quite negative or quite positive. These are CONTRADICTORY TRIALS. They are either low confidence + high EV + correct (right side of 0) or high confidence + low EV + incorrect (left side of 0). 

Why would you confidently pick a low EV option? Exploration, possibly. Why would you timidly pick a high EV option? High uncertainty, possibly. I HAVE to do some model comparison. 

At the very least, I get 4 kinds of choices: 

1. High confidence + high EV + no reward
2. High confidence + low EV + no reward
3. Low confidence + high EV + reward
4. Low confidence + low EV + reward

CPE is very rarely at 0. This means that high confidence + reward is kind of rare, as is low confidence + no reward (even rarer!)?

In [ ]:
f, ax = plt.subplots(1,1,figsize=[10,4])
sns.lineplot(data=learn_df[learn_df.participant=='MS036'], y='rpe_drift', x='trials_gamble', ci=95)
sns.lineplot(data=learn_df[learn_df.participant=='MS036'], y='rpe', x='trials_gamble', ci=95)
# plt.vlines([16, 32, 48, 64], ymin=-0.4, ymax=0.6, color='k')

In [ ]:
# Plotting code for smoothed time-resolved regression (complementary to binned analysis plotting above)

from mne.stats import permutation_cluster_test, permutation_cluster_1samp_test


# Note: For smoothed data, timepoints are already in ms and downsampled
# No need for win_len/sr conversion since ts values are already in ms

hpc_c = np.array([0.44314, 0.2902, 1])
amy_c = np.array([0.49412, 0, 0.90196])
ofc_c = np.array([1, 0.75686, 0.41569])
acc_c = np.array([0.7098, 0.27059, 0.68235])
dmpfc_c = np.array([0.40784, 0.01961, 0])
dlpfc_c = np.array([1, 0.06275, 0.06275])

fig, ax = plt.subplots(2, 3, figsize=(20, 10))

for event in ['feedback_start']:
    for predictor in ['zrpe']:
        for band in ['hfa']:
            # Load smoothed analysis results
            model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_smoothed_final.csv')   
        #     predictor = "C(rpe_cat, Treatment(reference='neutral'))[T.positive]"
            model_dfs['unique_label'] = model_dfs.apply(lambda x: x.subj + '_' + x.unique_label, axis=1)
        

            for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
                # plot the single-electrode encoding data 
                
                if region=='OFC':
                        color = ofc_c
                        ix = 0
                        jx = 0
                elif region=='HPC': 
                        color = hpc_c
                        ix = 0
                        jx = 2
                elif region=='AMY':
                        color = amy_c
                        ix = 1
                        jx = 2                       
                elif region=='dmPFC':
                        color = dmpfc_c
                        ix = 1
                        jx = 0
                elif region=='dlPFC':
                        color = dlpfc_c
                        ix = 1
                        jx = 1
                elif region=='ACC':
                        color = acc_c
                        ix = 0
                        jx = 1
                

                ts = model_dfs[(model_dfs.predictor==f'{predictor}') & (model_dfs.region==region)]['ts']
                
                # Get sorted unique timepoints - use this consistently for everything
                unique_ts = np.sort(model_dfs[(model_dfs.predictor==f'{predictor}') & (model_dfs.region==region)]['ts'].unique())
                
                # Calculate mean and standard error for each timepoint
                region_data = model_dfs[(model_dfs.predictor==f'{predictor}') & (model_dfs.region==region)]
                mean_beta = region_data.groupby('ts')['z_beta'].mean().loc[unique_ts].values
                se_beta = region_data.groupby('ts')['z_beta'].sem().loc[unique_ts].values
                
                # Plot line with error shading
                ax[ix, jx].plot(unique_ts, mean_beta, color=color, linewidth=1.5)
                ax[ix, jx].fill_between(unique_ts, mean_beta - se_beta, mean_beta + se_beta, 
                                       color=color, alpha=0.2)
                
                # For smoothed data, timepoints are already in ms, so adjust tick labels accordingly
                if event in ['cue_on', 'choice_time']:
                       # Use actual timepoints from data
                       xticks = unique_ts[::max(1, len(unique_ts)//5)]  # Sample ~5 ticks
                       xticklabels = [f'{int(t)}' for t in xticks]
                       xlim = [min(unique_ts) - 100, max(unique_ts) + 100]
                elif event == 'feedback_start':
                       # Use actual timepoints from data
                       xticks = unique_ts[::max(1, len(unique_ts)//7)]  # Sample ~7 ticks
                       xticklabels = [f'{int(t)}' for t in xticks]
                       xlim = [min(unique_ts) - 100, max(unique_ts) + 100]

                ax[ix, jx].set(xticks=xticks, 
                        xticklabels=xticklabels, 
                        xlim = xlim,
                        xlabel='time (ms)', ylabel=f'{predictor} (z)')
                sns.despine()
                
                # Run permutation test - reshape data using sorted unique timepoints
                T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
                        region_data.z_beta.values.reshape(-1, len(unique_ts)),
                        n_permutations=5000,
                        tail=0)
                
                if len(clusters)>0:
                        for i_c, c in enumerate(clusters):
                                if cluster_p_values[i_c] <= 0.05:
                                        # Convert cluster indices to actual time values
                                        # Clusters are indices into the sorted unique_ts array
                                        c_start_idx = max(0, clusters[i_c][0][0] - 1)
                                        c_end_idx = min(len(unique_ts) - 1, clusters[i_c][0][-1] + 1)
                                        # Get the actual time values at these indices
                                        c_ts = unique_ts[c_start_idx:c_end_idx+1]
                                        if len(c_ts) > 0:
                                            ax[ix, jx].fill_between(c_ts, 0.525, 0.6, color=color, alpha=0.5)

                ax[ix, jx].set(ylim=[-0.6, 0.6], title=region)
                ax[ix, jx].hlines(0, xmin=min(unique_ts), xmax=max(unique_ts), color='black', linestyle='--', linewidth=1)
                if (ix==0):
                       ax[ix, jx].set(xlabel=None, xticklabels=[])
                if (jx>0):
                       ax[ix, jx].set(ylabel=None, yticklabels=[])
                if ((ix==0) & (jx==0)) | ((ix==1) & (jx==0)):
                       pass
                else:
                       ax[ix, jx].set(ylabel=None)


plt.tight_layout()
plt.savefig(f'{plot_dir}/time_resolved_single_{band}_{event}_{predictor}_smoothed_final.pdf', dpi=300)
                                

In [ ]:
# FDR correct z_p values for each predictor for each unique_label,  across timestamps 'ts'
from statsmodels.stats.multitest import fdrcorrection

model_dfs['z_p_fdr'] = (
    model_dfs
    .groupby(['unique_label', 'predictor'])['z_p']
    .transform(lambda p: fdrcorrection(p.to_numpy(), alpha=0.05)[1])
)

model_dfs['sig_fdr'] = model_dfs['z_p_fdr'] < 0.05

In [ ]:
test_df = model_dfs[(model_dfs.predictor=='zrpe') & (model_dfs.region=='dmPFC')]

In [ ]:
test_df[test_df.z_p_fdr < 0.05].unique_label.unique().shape

In [ ]:
test_df.unique_label.unique().shape

In [ ]:
test_df = test_df[(test_df.subj=='MS036' )]

In [ ]:
test_df

In [ ]:
for i in test_df.unique_label.unique():
    plt.plot(test_df[test_df.unique_label==i].ts, test_df[test_df.unique_label==i].z_beta)
plt.legend()

In [ ]:
plt.plot(test_df.ts, test_df.z_beta)

In [ ]:
model_dfs

In [ ]:
from mne.stats import permutation_cluster_test, permutation_cluster_1samp_test


win_len = 50
sr = 500

hpc_c = np.array([0.44314, 0.2902, 1])
amy_c = np.array([0.49412, 0, 0.90196])
ofc_c = np.array([1, 0.75686, 0.41569])
acc_c = np.array([0.7098, 0.27059, 0.68235])
dmpfc_c = np.array([0.40784, 0.01961, 0])
dlpfc_c = np.array([1, 0.06275, 0.06275])

# Add horizontal lines
line_width = win_len * (1000/sr)

fig, ax = plt.subplots(2, 3, figsize=(20, 10))

for event in ['feedback_start']:
    for predictor in ['zrpe']:
        for band in ['hfa']:
            model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_final.csv')   

            for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
                # plot the single-electrode encoding data 
                
                if region=='OFC':
                        color = ofc_c
                        ix = 0
                        jx = 0
                elif region=='HPC': 
                        color = hpc_c
                        ix = 0
                        jx = 2
                elif region=='AMY':
                        color = amy_c
                        ix = 1
                        jx = 2                       
                elif region=='dmPFC':
                        color = dmpfc_c
                        ix = 1
                        jx = 0
                elif region=='dlPFC':
                        color = dlpfc_c
                        ix = 1
                        jx = 1
                elif region=='ACC':
                        color = acc_c
                        ix = 0
                        jx = 1
                

                ts = model_dfs[(model_dfs.predictor==f'{predictor}') & (model_dfs.region==region)]['ts']

                sns.barplot(x=ts,
                        y=model_dfs[(model_dfs.predictor==f'{predictor}') & (model_dfs.region==region)]['z_beta'],
                        width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se', ax=ax[ix, jx])
                if event in ['cue_on', 'choice_time']:
                       xticks = [-1, 4, 9, 14, 19]
                       xticklabels = [0, 250, 500, 750, 1000]
                       xlim = [-2, 20]
                elif event == 'feedback_start':
                       xticks=[-1, 4, 9, 14, 19, 24, 29]
                       xticklabels=[0, 250, 500, 750, 1000, 1250, 1500]
                       xlim = [-2, 30]

                ax[ix, jx].set(xticks=xticks, 
                        xticklabels=xticklabels, 
                        xlim = xlim,
                        xlabel='time-bin (ms)', ylabel=f'{predictor} (z)')
                sns.despine()
                
                T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
                        model_dfs[(model_dfs.predictor==f'{predictor}') & (model_dfs.region==region)].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
                        n_permutations=1000,
                        tail=0)
                
                if len(clusters)>0:
                        for i_c, c in enumerate(clusters):
                                if cluster_p_values[i_c] <= 0.05:
                                        c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
                                        ax[ix, jx].fill_between(c, 0.525, 0.6, color=color, alpha=0.5)

                ax[ix, jx].set(ylim=[-0.6, 0.6], title=region)
                if (ix==0):
                       ax[ix, jx].set(xlabel=None, xticklabels=[])
                if (jx>0):
                       ax[ix, jx].set(ylabel=None, yticklabels=[])
                if ((ix==0) & (jx==0)) | ((ix==1) & (jx==0)):
                       pass
                else:
                       ax[ix, jx].set(ylabel=None)


plt.tight_layout()
plt.savefig(f'{plot_dir}/time_resolved_single_{band}_{event}_{predictor}_final.pdf', dpi=300)
                                

In [ ]:
for event in ['feedback_start']:
    for predictor in ['zrpe']:
        for band in ['hfa']:
            model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_final.csv')   

            for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
                # plot the single-electrode encoding data 
                
                if region=='OFC':
                        color = ofc_c
                        ix = 0
                        jx = 0
                elif region=='HPC': 
                        color = hpc_c
                        ix = 0
                        jx = 2
                elif region=='AMY':
                        color = amy_c
                        ix = 1
                        jx = 2                       
                elif region=='dmPFC':
                        color = dmpfc_c
                        ix = 1
                        jx = 0
                elif region=='dlPFC':
                        color = dlpfc_c
                        ix = 1
                        jx = 1
                elif region=='ACC':
                        color = acc_c
                        ix = 0
                        jx = 1
                

                ts = model_dfs[(model_dfs.predictor==f'{predictor}:phit') & (model_dfs.region==region)]['ts']
                # Is there a difference between the two hemispheres?

                T_obs, clusters, cluster_p_values, H0 = permutation_cluster_test([model_dfs[(model_dfs.predictor==f'{predictor}:phit') & (model_dfs.region==region) & (model_dfs.hemi=='l')].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
                                                                                        model_dfs[(model_dfs.predictor==f'{predictor}:phit') & (model_dfs.region==region) & (model_dfs.hemi=='r')].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0])],
                        n_permutations=1000,
                        tail=0)
                

                if event in ['cue_on', 'choice_time']:
                       xticks = [-1, 4, 9, 14, 19]
                       xticklabels = [0, 250, 500, 750, 1000]
                       xlim = [-2, 20]
                elif event == 'feedback_start':
                       xticks=[-1, 4, 9, 14, 19, 24, 29]
                       xticklabels=[0, 250, 500, 750, 1000, 1250, 1500]
                       xlim = [-2, 30]
                
                if any(cluster_p_values<=0.05):
                       print(f'{region} {predictor} {event} {band} has significant difference between hemispheres')
                       fig, ax = plt.subplots(1, 2, figsize=(12, 5))

                       sns.barplot(x=ts, y=model_dfs[(model_dfs.predictor==f'{predictor}:phit') & (model_dfs.region==region) & (model_dfs.hemi=='l')]['z_beta'],
                                   width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se', ax=ax[0])
                       
                       sns.barplot(x=ts, y=model_dfs[(model_dfs.predictor==f'{predictor}:phit') & (model_dfs.region==region) & (model_dfs.hemi=='r')]['z_beta'],
                        width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se', ax=ax[1])
                       
                       ax[0].set(xticks=xticks, 
                        xticklabels=xticklabels, 
                        xlim = xlim,
                        xlabel='time-bin (ms)', ylabel=f'{predictor}:memory interaction (z)')
                       
                       ax[1].set(xticks=xticks, 
                        xticklabels=xticklabels, 
                        xlim = xlim,
                        xlabel='time-bin (ms)', ylabel=f'{predictor}:memory interaction (z)')
                       
                       if len(clusters)>0:
                             for i_c, c in enumerate(clusters):
                                   if cluster_p_values[i_c] <= 0.05:
                                        c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
                                        ax[0].fill_between(c, 0.525, 0.6, color=color, alpha=0.5)
                                        ax[1].fill_between(c, 0.525, 0.6, color=color, alpha=0.5)

                       plt.tight_layout()
                       plt.savefig(f'{plot_dir}/time_resolved_single_{band}_{event}_{predictor}_final_hemisplit_sig.pdf', dpi=300)
                       

Split hippocampus by anterior-posterior position

In [ ]:
model_dfs

In [ ]:
elec_dfs = []
for event in ['feedback_start']:
    for predictor in ['zrpe']:
        for band in ['hfa']:
            model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}_final.csv')   

            for region in ['HPC']:
                # plot the single-electrode encoding data 
                
                if region=='OFC':
                        color = ofc_c
                        ix = 0
                        jx = 0
                elif region=='HPC': 
                        color = hpc_c
                        ix = 0
                        jx = 2
                elif region=='AMY':
                        color = amy_c
                        ix = 1
                        jx = 2                       
                elif region=='dmPFC':
                        color = dmpfc_c
                        ix = 1
                        jx = 0
                elif region=='dlPFC':
                        color = dlpfc_c
                        ix = 1
                        jx = 1
                elif region=='ACC':
                        color = acc_c
                        ix = 0
                        jx = 1
                
                # load hippocampal coordinates
                day = 'Day1'
                for subj in model_dfs.subj.unique():
                        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/{day}_reref_elec_df')
                        elec_df['subj'] = subj
                        elec_df.rename(columns={'label':'unique_label'}, inplace=True)
                        elec_df['AP_pos'] = elec_df.apply(lambda x: 'posterior' if x.mni_x > -18 else 'anterior', axis=1)
                        elec_dfs.append(elec_df)
                elec_dfs = pd.concat(elec_dfs)


                plot_df = model_dfs[(model_dfs.predictor==f'{predictor}:phit') & (model_dfs.region==region)]
                ts = plot_df['ts']

                plot_df = plot_df.merge(elec_dfs, on=['subj', 'unique_label'])



                # split via A-P position

                # Is there a difference between the two hemispheres?

                T_obs, clusters, cluster_p_values, H0 = permutation_cluster_test([plot_df[(plot_df.AP_pos=='anterior')].z_beta.values.reshape(-1, plot_df.ts.unique().shape[0]),
                                                                                        plot_df[(plot_df.AP_pos=='posterior')].z_beta.values.reshape(-1, plot_df.ts.unique().shape[0])],
                        n_permutations=1000,
                        tail=0)
                

                if event in ['cue_on', 'choice_time']:
                       xticks = [-1, 4, 9, 14, 19]
                       xticklabels = [0, 250, 500, 750, 1000]
                       xlim = [-2, 20]
                elif event == 'feedback_start':
                       xticks=[-1, 4, 9, 14, 19, 24, 29]
                       xticklabels=[0, 250, 500, 750, 1000, 1250, 1500]
                       xlim = [-2, 30]
                
                if any(cluster_p_values<=0.05):
                       print(f'{region} {predictor} {event} {band} has significant difference between hemispheres')
                       fig, ax = plt.subplots(1, 2, figsize=(12, 5))

                       sns.barplot(x=ts, y=plot_df[(plot_df.AP_pos=='anterior')]['z_beta'],
                                   width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se', ax=ax[0])
                       
                       sns.barplot(x=ts, y=plot_df[(plot_df.AP_pos=='posterior')]['z_beta'],
                        width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se', ax=ax[1])
                       
                       ax[0].set(xticks=xticks, 
                        xticklabels=xticklabels, 
                        xlim = xlim,
                        xlabel='time-bin (ms)', ylabel=f'{predictor}:memory interaction (z)')
                       
                       ax[1].set(xticks=xticks, 
                        xticklabels=xticklabels, 
                        xlim = xlim,
                        xlabel='time-bin (ms)', ylabel=f'{predictor}:memory interaction (z)')
                       
                       if len(clusters)>0:
                             for i_c, c in enumerate(clusters):
                                   if cluster_p_values[i_c] <= 0.05:
                                        c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
                                        ax[0].fill_between(c, 0.525, 0.6, color=color, alpha=0.5)
                                        ax[1].fill_between(c, 0.525, 0.6, color=color, alpha=0.5)

                       plt.tight_layout()
                       plt.savefig(f'{plot_dir}/time_resolved_single_{band}_{event}_{predictor}_final_AP_split_sig.pdf', dpi=300)
                       

5/15/2025: There is no difference between more anterior and posterior 

In [ ]:
plot_df[(plot_df.AP_pos=='anterior')].z_beta.values.reshape(-1, plot_df.ts.unique().shape[0])

In [ ]:
elec_dfs.mni_y

In [ ]:
plt.hist(elec_dfs[elec_dfs.salman_region=='HPC'].mni_y)

In [ ]:
# event = 'choice_time'
# # model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_zpmcontrol.csv')   
# model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}.csv')   


# win_len = 50
# sr = 500

# # Add horizontal lines
# line_width = win_len * (1000/sr)

# for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
#        # plot the single-electrode encoding data 
#        fig, ax = plt.subplots(1, 1, figsize=(8, 5))
#        if region=='OFC':
#               color = ofc_c
#        elif region=='HPC': 
#               color = hpc_c
#        elif region=='AMY':
#               color = amy_c
#        elif region=='dmPFC':
#               color = dmpfc_c
#        elif region=='dlPFC':
#               color = dlpfc_c
#        elif region=='ACC':
#               color = acc_c

#        ts = model_dfs[(model_dfs.predictor=='zQ') & (model_dfs.region==region)]['ts']
#        sns.barplot(x=ts,
#               y=model_dfs[(model_dfs.predictor=='zQ') & (model_dfs.region==region)]['z_beta'],
#               width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se')
#        ax.set(xticks=[-1, 4, 9, 14, 19,], 
#               xticklabels=[0, 250, 500, 750, 1000], 
#               xlim = [-2, 20],
#               xlabel='time-bin (ms)', ylabel='rpe:memory interaction (z)')
#        sns.despine()
       
#        T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
#         model_dfs[(model_dfs.predictor=="zQ") & (model_dfs.region==region)].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
#         n_permutations=1000,
#         tail=0)
       
#        if len(clusters)>0:
#               for i_c, c in enumerate(clusters):
#                      if cluster_p_values[i_c] <= 0.05:
#                             c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
#                             ax.fill_between(c, plt.ylim()[0], plt.ylim()[1], color=color, alpha=0.5)

#        ax.set(ylim=[-0.6, 0.6], title=region)


#        plt.savefig(f'/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots/time_resolved_single_{event}_{band}_{region}.pdf', dpi=300)


In [ ]:
# progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
# freqs = np.logspace(*np.log10([2, 200]), num=30)
# event = 'feedback_start'

# freq_dict = {'theta':[2, 8], 
#             'alpha':[8, 13],
#             'beta': [13, 30], 
#             'hfa': [70, 200]}

# rois = ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']
# day = 'Day1'
# y='tfr'
# formula='tfr ~ 1 + zQ*phit'
# win_len = 50
# slide_len = 25
# sr=500

# combined_df = combined_df[combined_df.condition=='Day1']

# for band in ['hfa']:
#     model_dfs = []
#     for subj in subj_ids: 
#         # normalize variables
#         combined_df.loc[combined_df.participant==subj, 
#                         'zrpe'] = (combined_df.loc[combined_df.participant==subj, 
#                                                                                 'rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
#                                                                                                         'rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
#                                                                                                                                             'rpe']))
#         combined_df.loc[combined_df.participant==subj, 
#                 'zpm'] = (combined_df.loc[combined_df.participant==subj, 
#                                                                         'DPRIME'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
#                                                                                                 'DPRIME'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
#                                                                                                                                     'DPRIME']))
#         combined_df['phit'] = combined_df.apply(lambda x: 0.5 if x.hits==True else -0.5, axis=1)

#         # 2/16/25: recover the latent Q-values and fit separate from reward

#         combined_df['Q'] = combined_df.rpe + combined_df.reward

#         combined_df.loc[combined_df.participant==subj, 
#                 'zQ'] = (combined_df.loc[combined_df.participant==subj, 
#                                                                         'Q'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
#                                                                                                 'Q'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
#                                                                                                                                     'Q']))

#         combined_df['preward'] = combined_df.apply(lambda x: 0.5 if x.reward==1 else -0.5, axis=1)

#         if event == 'cue_on':
#             trial_var = 'trials_mem'
#         else:
#             trial_var = 'trials_gamble'

#         if event == 'cue_on':
#             regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'zrpe', 'zpm', 'phit', f'{trial_var}', 'zQ', 'recog_rt']]
#         else:
#             regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'zrpe', 'zpm', 'phit', f'{trial_var}', 'zQ']]


#         elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/{day}_reref_elec_df')
#         filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/scratch/TFR'
#         tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')

#         # get all the labels in elec_df for which salman region is in rois
#         roi_index = elec_df.loc[elec_df.salman_region.isin(rois), 'label'].values.tolist()

#         if len(roi_index)==0:
#             continue

#         # get the average time series across frequenceis for all channels in the roi
#         timeseries = np.nanmean(tfr_data.copy().pick(roi_index).data[:, :,
#                                                                     np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
#                                                                     :], axis=2)
#         # define slices for smoothing timeseries: 
#         slices = np.lib.stride_tricks.sliding_window_view(np.arange(tfr_data.data.shape[-1]), win_len)[::slide_len]
#         midpoints = np.ceil(np.mean(slices, axis=1))
#         ts = midpoints * (1000/sr)

#         sig = np.zeros([timeseries.shape[0], timeseries.shape[1], slices.shape[0]])
#         for stride in range(slices.shape[0]):
#             sig[:, :, stride] = np.nanmean(timeseries[:, :, slices[stride]], axis=-1)

#         for j, ch in enumerate(roi_index):
#             for i in range(len(ts)):
#                 ch_df = regressors.copy()
#                 ch_df['ts'] = ts[i]
#                 ch_df.rename(columns={f'{trial_var}':'trial'}, inplace=True)
#                 ch_df.dropna(subset=['zrpe'], inplace=True)
#                 ch_df.sort_values(by=['trial'], inplace=True)
#                 trial_ix = ch_df.trial.values.astype(int) - 1
#                 ch_df['tfr'] = sig[trial_ix, j, i]
#                 ch_df['unique_label'] = ch
#                 ch_df['region'] = elec_df[elec_df.label==ch].salman_region.values[0] 
#                 # ch_dfs.append(ch_df)
            
#                 results = statistics_utils.permutation_regression_zscore(ch_df,
#                         formula=formula,
#                         n_permutations=250, 
#                         plot_res=False)
#                 results['ts'] = i
#                 results['subj'] = subj
#                 results['unique_label'] = ch
#                 results['region'] = elec_df[elec_df.label==ch].salman_region.values[0]
#                 model_dfs.append(results)
        
#     model_dfs = pd.concat(model_dfs)
#     for ts_i in model_dfs.ts.unique():
#         model_dfs.loc[model_dfs.ts==ts_i, 'ts'] = midpoints[ts_i] * (1000/sr)
#     model_dfs.to_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_zQ.csv', index=False)   

In [ ]:
event = 'feedback_start'
# model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_zpmcontrol.csv')   
model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_zRT_rpe.csv')   


win_len = 50
sr = 500

# Add horizontal lines
line_width = win_len * (1000/sr)

for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
       # plot the single-electrode encoding data 
       fig, ax = plt.subplots(1, 1, figsize=(8, 5))
       if region=='OFC':
              color = ofc_c
       elif region=='HPC': 
              color = hpc_c
       elif region=='AMY':
              color = amy_c
       elif region=='dmPFC':
              color = dmpfc_c
       elif region=='dlPFC':
              color = dlpfc_c
       elif region=='ACC':
              color = acc_c

       ts = model_dfs[(model_dfs.predictor=='zRT_rpe:phit') & (model_dfs.region==region)]['ts']
       sns.barplot(x=ts,
              y=model_dfs[(model_dfs.predictor=='zRT_rpe:phit') & (model_dfs.region==region)]['z_beta'],
              width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se')
       ax.set(xticks=[-1, 4, 9, 14, 19, 24, 29], 
              xticklabels=[0, 250, 500, 750, 1000, 1250, 1500], 
              xlim = [-2, 30],
              xlabel='time-bin (ms)', ylabel='zRT_rpe:memory interaction (z)')
       sns.despine()
       
       T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
        model_dfs[(model_dfs.predictor=="zRT_rpe:phit") & (model_dfs.region==region)].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
        n_permutations=1000,
        tail=0)
       
       if len(clusters)>0:
              for i_c, c in enumerate(clusters):
                     if cluster_p_values[i_c] <= 0.05:
                            c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
                            ax.fill_between(c, plt.ylim()[0], plt.ylim()[1], color=color, alpha=0.5)

       ax.set(ylim=[-0.6, 0.6], title=region)


       plt.savefig(f'{plot_dir}/time_resolved_single_{event}_{band}_{region}_zRT_rpe.pdf', dpi=300)


In [ ]:
event = 'feedback_start'
# model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_zpmcontrol.csv')   
model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_zQ.csv')   


win_len = 50
sr = 500

# Add horizontal lines
line_width = win_len * (1000/sr)

for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
       # plot the single-electrode encoding data 
       fig, ax = plt.subplots(1, 1, figsize=(8, 5))
       if region=='OFC':
              color = ofc_c
       elif region=='HPC': 
              color = hpc_c
       elif region=='AMY':
              color = amy_c
       elif region=='dmPFC':
              color = dmpfc_c
       elif region=='dlPFC':
              color = dlpfc_c
       elif region=='ACC':
              color = acc_c

       ts = model_dfs[(model_dfs.predictor=='zQ:phit') & (model_dfs.region==region)]['ts']
       sns.barplot(x=ts,
              y=model_dfs[(model_dfs.predictor=='zQ:phit') & (model_dfs.region==region)]['z_beta'],
              width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se')
       ax.set(xticks=[-1, 4, 9, 14, 19, 24, 29], 
              xticklabels=[0, 250, 500, 750, 1000, 1250, 1500], 
              xlim = [-2, 30],
              xlabel='time-bin (ms)', ylabel='rpe:memory interaction (z)')
       sns.despine()
       
       T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
        model_dfs[(model_dfs.predictor=="zQ:phit") & (model_dfs.region==region)].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
        n_permutations=1000,
        tail=0)
       
       if len(clusters)>0:
              for i_c, c in enumerate(clusters):
                     if cluster_p_values[i_c] <= 0.05:
                            c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
                            ax.fill_between(c, plt.ylim()[0], plt.ylim()[1], color=color, alpha=0.5)

       ax.set(ylim=[-0.6, 0.6], title=region)


       plt.savefig(f'{plot_dir}/time_resolved_single_{event}_{band}_{region}_zQ.pdf', dpi=300)


In [ ]:
event = 'feedback_start'
# model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_zpmcontrol.csv')   
model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_preward.csv')   


win_len = 50
sr = 500

# Add horizontal lines
line_width = win_len * (1000/sr)

for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
       # plot the single-electrode encoding data 
       fig, ax = plt.subplots(1, 1, figsize=(8, 5))
       if region=='OFC':
              color = ofc_c
       elif region=='HPC': 
              color = hpc_c
       elif region=='AMY':
              color = amy_c
       elif region=='dmPFC':
              color = dmpfc_c
       elif region=='dlPFC':
              color = dlpfc_c
       elif region=='ACC':
              color = acc_c

       ts = model_dfs[(model_dfs.predictor=='preward:phit') & (model_dfs.region==region)]['ts']
       sns.barplot(x=ts,
              y=model_dfs[(model_dfs.predictor=='preward:phit') & (model_dfs.region==region)]['z_beta'],
              width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se')
       ax.set(xticks=[-1, 4, 9, 14, 19, 24, 29], 
              xticklabels=[0, 250, 500, 750, 1000, 1250, 1500], 
              xlim = [-2, 30],
              xlabel='time-bin (ms)', ylabel='reward:memory interaction (z)')
       sns.despine()
       
       T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
        model_dfs[(model_dfs.predictor=="preward:phit") & (model_dfs.region==region)].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
        n_permutations=1000,
        tail=0)
       
       if len(clusters)>0:
              for i_c, c in enumerate(clusters):
                     if cluster_p_values[i_c] <= 0.05:
                            c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
                            ax.fill_between(c, plt.ylim()[0], plt.ylim()[1], color=color, alpha=0.5)

       ax.set(ylim=[-0.6, 0.6], title=region)


       plt.savefig(f'{plot_dir}/time_resolved_single_{event}_{band}_{region}_preward.pdf', dpi=300)
# 

In [ ]:
model_dfs

In [ ]:
event = 'feedback_start'
# model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_zpmcontrol.csv')   
model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_hemi.csv')   


win_len = 50
sr = 500

# Add horizontal lines
line_width = win_len * (1000/sr)

for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
       # plot the single-electrode encoding data 
       fig, ax = plt.subplots(1, 1, figsize=(8, 5))
       if region=='OFC':
              color = ofc_c
       elif region=='HPC': 
              color = hpc_c
       elif region=='AMY':
              color = amy_c
       elif region=='dmPFC':
              color = dmpfc_c
       elif region=='dlPFC':
              color = dlpfc_c
       elif region=='ACC':
              color = acc_c

       ts = model_dfs[(model_dfs.predictor=='zrpe:phit') & (model_dfs.region==region)]['ts']
       sns.barplot(x=ts,
              y=model_dfs[(model_dfs.predictor=='zrpe:phit') & (model_dfs.region==region)]['z_beta'],
              width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se')
       ax.set(xticks=[-1, 4, 9, 14, 19, 24, 29], 
              xticklabels=[0, 250, 500, 750, 1000, 1250, 1500], 
              xlim = [-2, 30],
              xlabel='time-bin (ms)', ylabel='rpe:memory interaction (z)')
       sns.despine()
       
       T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
        model_dfs[(model_dfs.predictor=="zrpe:phit") & (model_dfs.region==region)].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
        n_permutations=1000,
        tail=0)
       
       if len(clusters)>0:
              for i_c, c in enumerate(clusters):
                     if cluster_p_values[i_c] <= 0.05:
                            c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
                            ax.fill_between(c, plt.ylim()[0], plt.ylim()[1], color=color, alpha=0.5)

       ax.set(ylim=[-0.6, 0.6], title=region)


       plt.savefig(f'{plot_dir}/time_resolved_single_{event}_{band}_{region}_zrpe.pdf', dpi=300)
       # plt.savefig(f'/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots/time_resolved_single_{event}_{band}_{region}_preward.pdf', dpi=300)
# 

## Cue


2/27/25: Run the multinomial regression for rewarded vs. unrewarded stimuli only

In [ ]:
combined_df.recog_rt.mean()

In [ ]:
progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'cue_on'

freq_dict = {'theta':[4, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

rois = ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']
day = 'Day1'
y='tfr'
# formula = "tfr ~ 1 + preward"

# formula="tfr ~ 1 + recog_rt_log + zrpe*phit"
# C(memory_outcome, Treatment(reference='hits'))"
# formula="tfr ~ 1 + recog_rt_log + C(recog_choice) + good_memory"
# formula="tfr ~ 1 + C(memory_outcome, Treatment(reference='false_alarms'))"
# formula = "tfr ~ 1 + preward"

win_len = 50
slide_len = 25
sr=500

combined_df = combined_df[combined_df.condition=='Day1']
for predictor in ['zRT_rpe']:
    formula = f"tfr ~ 1 + {predictor}*phit"
    for band in ['hfa']:
        model_dfs = []
        for subj in subj_ids: 
 
            combined_df['good_memory'] = combined_df.apply(lambda x: 0.5 if (x.memory_outcome=='hits') | (x.memory_outcome=='correct_rejections') else -0.5, axis=1)
            combined_df['old'] = combined_df.apply(lambda x: 0.5 if (x.memory_outcome=='hits') | (x.memory_outcome=='false_alarms') else -0.5, axis=1)

            if event == 'cue_on':
                trial_var = 'trials_mem'
            else:
                trial_var = 'trials_gamble'

            if event in ['cue_on', 'recog_time']:
                regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'zrpe', 'zpm', 'phit', f'{trial_var}', 'preward', 'recog_rt', 'memory_outcome', 'good_memory', 'recog_choice']]
            else:
                regressors = combined_df.loc[combined_df.participant==subj, ['participant', 'zrpe', 'zpm', 'phit', f'{trial_var}', 'preward']]

            regressors['recog_rt_log'] = np.log10(regressors.recog_rt)

            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/{day}_reref_elec_df')
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/scratch/TFR'
            tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')

            # get all the labels in elec_df for which salman region is in rois
            roi_index = elec_df.loc[elec_df.salman_region.isin(rois), 'label'].values.tolist()

            if len(roi_index)==0:
                continue

            # get the average time series across frequenceis for all channels in the roi
            timeseries = np.nanmean(tfr_data.copy().pick(roi_index).data[:, :,
                                                                        np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                                        :], axis=2)
            # define slices for smoothing timeseries: 
            slices = np.lib.stride_tricks.sliding_window_view(np.arange(tfr_data.data.shape[-1]), win_len)[::slide_len]
            midpoints = np.ceil(np.mean(slices, axis=1))
            ts = midpoints * (1000/sr)

            sig = np.zeros([timeseries.shape[0], timeseries.shape[1], slices.shape[0]])
            for stride in range(slices.shape[0]):
                sig[:, :, stride] = np.nanmean(timeseries[:, :, slices[stride]], axis=-1)

            for j, ch in enumerate(roi_index):
                for i in range(len(ts)):
                    ch_df = regressors.copy()
                    ch_df['ts'] = ts[i]
                    ch_df.rename(columns={f'{trial_var}':'trial'}, inplace=True)
                    # ch_df.dropna(subset=['zrpe'], inplace=True)
                    ch_df.sort_values(by=['trial'], inplace=True)
                    trial_ix = ch_df.trial.values.astype(int) - 1
                    ch_df['tfr'] = sig[trial_ix, j, i]
                    ch_df['unique_label'] = ch
                    ch_df['region'] = elec_df[elec_df.label==ch].salman_region.values[0] 
                    # ch_dfs.append(ch_df)

                    # # DROP ANY trials with too slow or too fast rt
                    ch_df = ch_df[((ch_df.recog_rt>=1) & (ch_df.recog_rt<=10))]
                
                    results = statistics_utils.permutation_regression_zscore(ch_df,
                            formula=formula,
                            n_permutations=250, 
                            plot_res=False)
                    results['ts'] = i
                    results['subj'] = subj
                    results['unique_label'] = ch
                    results['region'] = elec_df[elec_df.label==ch].salman_region.values[0]
                    model_dfs.append(results)
            
        model_dfs = pd.concat(model_dfs)
        for ts_i in model_dfs.ts.unique():
            model_dfs.loc[model_dfs.ts==ts_i, 'ts'] = midpoints[ts_i] * (1000/sr)
        
        model_dfs.to_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_{predictor}.csv', index=False)   

    # model_dfs.to_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_goodvbadmemory_rt_choice.csv', index=False)   

    # model_dfs.to_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_memory_rt.csv', index=False)   


In [ ]:
model_dfs

In [ ]:
# drop any rows from ch_df if reward==1 AND memory_outcome==hits
model_dfs

In [ ]:
event = 'cue_on'
# model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_zpmcontrol.csv')   
model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_zrpememory_rt.csv')   
model_dfs.loc[model_dfs.predictor.str.contains('memory_outcome'), 'z_beta'] = -model_dfs.loc[model_dfs.predictor.str.contains('memory_outcome'), 'z_beta'] 

win_len = 50
sr = 500

# Add horizontal lines
line_width = win_len * (1000/sr)

for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
       # plot the single-electrode encoding data 
       fig, ax = plt.subplots(1, 1, figsize=(8, 5), sharey=True)

       if region=='OFC':
              color = ofc_c
       elif region=='HPC': 
              color = hpc_c
       elif region=='AMY':
              color = amy_c
       elif region=='dmPFC':
              color = dmpfc_c
       elif region=='dlPFC':
              color = dlpfc_c
       elif region=='ACC':
              color = acc_c

       
       
       ts = model_dfs[(model_dfs.predictor=='zrpe:phit') & (model_dfs.region==region)]['ts']
       sns.barplot(x=ts,
              y=model_dfs[(model_dfs.predictor=='zrpe:phit') & (model_dfs.region==region)]['z_beta'],
              width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se', ax=ax)
       ax.set(xticks=[-1, 4, 9, 14, 19], 
              xticklabels=[0, 250, 500, 750, 1000], 
              xlim = [-2, 20],
              xlabel='time-bin (ms)', ylabel='rpe:memory interaction (z)')
       sns.despine()


       T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
       model_dfs[(model_dfs.predictor=='zrpe:phit') & (model_dfs.region==region)].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
       n_permutations=1000,
       tail=0)
       
       if len(clusters)>0:
              for i_c, c in enumerate(clusters):
                     if cluster_p_values[i_c] <= 0.05:
                            c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
                            ax.fill_between(c, plt.ylim()[0], plt.ylim()[1], color=color, alpha=0.5)

       ax.set(ylim=[-0.6, 0.6], title=f'{region}:good vs. bad memories')


       plt.savefig(f'{plot_dir}/time_resolved_single_{event}_{band}_{region}_zrpe_phit.pdf', dpi=300)


In [ ]:
model_dfs

In [ ]:
event = 'cue_on'
band = 'hfa'
# model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_zpmcontrol.csv')   
model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_memory_rt.csv')  

# When comparing to hits, we want to flip the sign of the memory_outcome betas
model_dfs.loc[model_dfs.predictor.str.contains('memory_outcome'), 'z_beta'] = -model_dfs.loc[model_dfs.predictor.str.contains('memory_outcome'), 'z_beta'] 

win_len = 50
sr = 500

# Add horizontal lines
line_width = win_len * (1000/sr)

for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
       # plot the single-electrode encoding data 
       fig, ax = plt.subplots(1, 3, figsize=(24, 5), sharey=True)

       if region=='OFC':
              color = ofc_c
       elif region=='HPC': 
              color = hpc_c
       elif region=='AMY':
              color = amy_c
       elif region=='dmPFC':
              color = dmpfc_c
       elif region=='dlPFC':
              color = dlpfc_c
       elif region=='ACC':
              color = acc_c

       
       for ix, memory_outcome in enumerate(['correct_rejections', 'false_alarms', 'misses']):
              
              ts = model_dfs[(model_dfs.predictor==f"C(memory_outcome, Treatment(reference='hits'))[T.{memory_outcome}]") & (model_dfs.region==region)]['ts']
              sns.barplot(x=ts,
                     y=model_dfs[(model_dfs.predictor==f"C(memory_outcome, Treatment(reference='hits'))[T.{memory_outcome}]") & (model_dfs.region==region)]['z_beta'],
                     width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se', ax=ax[ix])
              ax[ix].set(xticks=[-1, 4, 9, 14, 19], 
                     xticklabels=[0, 250, 500, 750, 1000], 
                     xlim = [-2, 20],
                     xlabel='time-bin (ms)', ylabel='memory weight (z)')
              sns.despine()

       
              T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
              model_dfs[(model_dfs.predictor==f"C(memory_outcome, Treatment(reference='hits'))[T.{memory_outcome}]") & (model_dfs.region==region)].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
              n_permutations=1000,
              tail=0)
              
              if len(clusters)>0:
                     for i_c, c in enumerate(clusters):
                            if cluster_p_values[i_c] <= 0.05:
                                   c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
                                   ax[ix].fill_between(c, plt.ylim()[0], plt.ylim()[1], color=color, alpha=0.5)

              ax[ix].set(ylim=[-1, 1], title=f'{region}:hits vs. {memory_outcome}')


       plt.savefig(f'{plot_dir}/time_resolved_single_{event}_{band}_{region}_hitvall_nort.pdf', dpi=300)

In [ ]:
model_dfs

In [ ]:
model_dfs.predictor.unique()

In [ ]:
model_dfs[(model_dfs.predictor==f"amy_c = [T.{memory_outcome}]") & (model_dfs.region==region)]['ts']

In [ ]:
model_dfs

In [ ]:
# event = 'cue_on'
# # model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_zpmcontrol.csv')   
# model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}_goodvbad_memory.csv')   
# model_dfs.loc[model_dfs.predictor.str.contains('memory_outcome'), 'z_beta'] = -model_dfs.loc[model_dfs.predictor.str.contains('memory_outcome'), 'z_beta'] 

# win_len = 50
# sr = 500

# # Add horizontal lines
# line_width = win_len * (1000/sr)

# for region in ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']:
#        # plot the single-electrode encoding data 
#        fig, ax = plt.subplots(1, 3, figsize=(24, 5), sharey=True)

#        if region=='OFC':
#               color = ofc_c
#        elif region=='HPC': 
#               color = hpc_c
#        elif region=='AMY':
#               color = amy_c
#        elif region=='dmPFC':
#               color = dmpfc_c
#        elif region=='dlPFC':
#               color = dlpfc_c
#        elif region=='ACC':
#               color = acc_c

       
#        for ix, memory_outcome in enumerate(['misses', 'false_alarms', 'correct_rejections']):
              
#               ts = model_dfs[(model_dfs.predictor==f"C(memory_outcome, Treatment(reference='hits'))[T.{memory_outcome}]") & (model_dfs.region==region)]['ts']
#               sns.barplot(x=ts,
#                      y=model_dfs[(model_dfs.predictor==f"C(memory_outcome, Treatment(reference='hits'))[T.{memory_outcome}]") & (model_dfs.region==region)]['z_beta'],
#                      width=2, color=color, alpha=0.2, edgecolor='black', linewidth=1, errorbar='se', ax=ax[ix])
#               ax[ix].set(xticks=[-1, 4, 9, 14, 19], 
#                      xticklabels=[0, 250, 500, 750, 1000], 
#                      xlim = [-2, 20],
#                      xlabel='time-bin (ms)', ylabel='memory weight (z)')
#               sns.despine()

       
#               T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
#               model_dfs[(model_dfs.predictor==f"C(memory_outcome, Treatment(reference='hits'))[T.{memory_outcome}]") & (model_dfs.region==region)].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
#               n_permutations=1000,
#               tail=0)
              
#               if len(clusters)>0:
#                      for i_c, c in enumerate(clusters):
#                             if cluster_p_values[i_c] <= 0.05:
#                                    c = np.arange(clusters[i_c][0][0]-1, clusters[i_c][0][-1]+1)
#                                    ax[ix].fill_between(c, plt.ylim()[0], plt.ylim()[1], color=color, alpha=0.5)

#               ax[ix].set(ylim=[-0.6, 0.6], title=f'{region}:hits vs. {memory_outcome}')


#        plt.savefig(f'/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots/time_resolved_single_{event}_{band}_{region}.pdf', dpi=300)


1. HPC increases HFA during hits compared to misses around ~300-650 ms after cue presentation.

2. HPC and dmPFC increase HFA during hits compared to false alarms around ~650 ms after cue presentation. 

3. Prefrontal cortex increases HFA during hits compared to correct rejections around ~ 500 ms after cue presentation. 

In [ ]:
model_dfs[(model_dfs.predictor==f"C(memory_outcome, Treatment(reference='hits'))[T.{memory_outcome}]") & (model_dfs.region==region)]['ts'].values[2,3]

In [ ]:
memory_outcome

In [ ]:
plot_df

In [ ]:
band = 'hfa'
model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}.csv')
                 

Now let's load the single-electrode results and characterize their "biomarker rank"

In [ ]:
OFC_df_single_elec = model_dfs[(model_dfs.region=='OFC')]
OFC_df_single_elec['unique_label'] = OFC_df_single_elec.subj + '_' + OFC_df_single_elec.unique_label.astype(str)

# compute the average z-score for zrpe:phit for sig_windows in ts
sig_windows = np.concatenate([np.arange(300, 550, 50), np.arange(900, 1050, 50), np.arange(1250, 1500, 50)])
sig_OFC_z = OFC_df_single_elec.loc[OFC_df_single_elec.ts.isin(sig_windows) & (OFC_df_single_elec.predictor=='zrpe:phit')].groupby(['unique_label', 'subj']).mean(numeric_only=True).reset_index()
# nsig_OFC_z = OFC_df_single_elec.loc[~OFC_df_single_elec.ts.isin(sig_windows) & (OFC_df_single_elec.predictor=='zrpe:phit')].groupby('unique_label').mean(numeric_only=True).reset_index()
# rank order the electrodes by their sig_OFC_z.z_beta within subj
sig_OFC_z['rank'] = sig_OFC_z.groupby('subj')['z_beta'].rank(ascending=False)
# rank order the electrodes overall by their sig_OFC_z.z_beta
sig_OFC_z['overall_rank'] = sig_OFC_z['z_beta'].rank(ascending=False)

sig_OFC_z[['unique_label','z_beta','rank', 'overall_rank']].to_csv(f'{load_dir}/OFC_biomarker_rank.csv', index=False)



In [ ]:
HPC_df_single_elec = model_dfs[(model_dfs.region=='HPC')]
HPC_df_single_elec['unique_label'] = HPC_df_single_elec.subj + '_' + HPC_df_single_elec.unique_label.astype(str)

# compute the average z-score for zrpe:phit for sig_windows in ts
sig_windows = np.concatenate([np.arange(600, 1000, 50), np.arange(1050, 1250, 50)])
sig_HPC_z = HPC_df_single_elec.loc[HPC_df_single_elec.ts.isin(sig_windows) & (HPC_df_single_elec.predictor=='zrpe:phit')].groupby(['unique_label', 'subj']).mean(numeric_only=True).reset_index()
# nsig_OFC_z = HPC_df_single_elec.loc[~HPC_df_single_elec.ts.isin(sig_windows) & (HPC_df_single_elec.predictor=='zrpe:phit')].groupby('unique_label').mean(numeric_only=True).reset_index()
# rank order the electrodes by their sig_OFC_z.z_beta within subj
sig_HPC_z['rank'] = sig_HPC_z.groupby('subj')['z_beta'].rank(ascending=True)

sig_HPC_z['overall_rank'] = sig_HPC_z['z_beta'].rank(ascending=True)


sig_HPC_z[['unique_label','z_beta','rank', 'overall_rank']].to_csv(f'{load_dir}/HPC_biomarker_rank.csv', index=False)



In [ ]:
AMY_df_single_elec = model_dfs[(model_dfs.region=='AMY')]
AMY_df_single_elec['unique_label'] = AMY_df_single_elec.subj + '_' + AMY_df_single_elec.unique_label.astype(str)

# compute the average z-score for zrpe:phit for sig_windows in ts
sig_windows = np.concatenate([np.arange(300, 450, 50), np.arange(1200, 1350, 50)])
sig_AMY_z = AMY_df_single_elec.loc[AMY_df_single_elec.ts.isin(sig_windows) & (AMY_df_single_elec.predictor=='zrpe:phit')].groupby(['unique_label', 'subj']).mean(numeric_only=True).reset_index()
# nsig_OFC_z = HPC_df_single_elec.loc[~HPC_df_single_elec.ts.isin(sig_windows) & (HPC_df_single_elec.predictor=='zrpe:phit')].groupby('unique_label').mean(numeric_only=True).reset_index()
# rank order the electrodes by their sig_OFC_z.z_beta within subj
sig_AMY_z['rank'] = sig_AMY_z.groupby('subj')['z_beta'].rank(ascending=True)

sig_AMY_z['overall_rank'] = sig_AMY_z['z_beta'].rank(ascending=True)


sig_AMY_z[['unique_label','z_beta','rank', 'overall_rank']].to_csv(f'{load_dir}/AMY_biomarker_rank.csv', index=False)



In [ ]:
dmPFC_df_single_elec = model_dfs[(model_dfs.region=='dmPFC')]
dmPFC_df_single_elec['unique_label'] = dmPFC_df_single_elec.subj + '_' + dmPFC_df_single_elec.unique_label.astype(str)

# compute the average z-score for zrpe:phit for sig_windows in ts
sig_windows = np.arange(1200, 1500, 50)
sig_dmPFC_z = dmPFC_df_single_elec.loc[dmPFC_df_single_elec.ts.isin(sig_windows) & (dmPFC_df_single_elec.predictor=='zrpe:phit')].groupby(['unique_label', 'subj']).mean(numeric_only=True).reset_index()
# nsig_OFC_z = HPC_df_single_elec.loc[~HPC_df_single_elec.ts.isin(sig_windows) & (HPC_df_single_elec.predictor=='zrpe:phit')].groupby('unique_label').mean(numeric_only=True).reset_index()
# rank order the electrodes by their sig_OFC_z.z_beta within subj
sig_dmPFC_z['rank'] = sig_dmPFC_z.groupby('subj')['z_beta'].rank(ascending=False)

sig_dmPFC_z['overall_rank'] = sig_dmPFC_z['z_beta'].rank(ascending=False)


sig_dmPFC_z[['unique_label','z_beta','rank', 'overall_rank']].to_csv(f'{load_dir}/dmPFC_biomarker_rank.csv', index=False)



In [ ]:
dlPFC_df_single_elec = model_dfs[(model_dfs.region=='dlPFC')]
dlPFC_df_single_elec['unique_label'] = dlPFC_df_single_elec.subj + '_' + dlPFC_df_single_elec.unique_label.astype(str)

# compute the average z-score for zrpe:phit for sig_windows in ts
sig_windows = np.arange(300, 450, 50)
sig_dlPFC_z = dlPFC_df_single_elec.loc[dlPFC_df_single_elec.ts.isin(sig_windows) & (dlPFC_df_single_elec.predictor=='zrpe:phit')].groupby(['unique_label', 'subj']).mean(numeric_only=True).reset_index()
# nsig_OFC_z = HPC_df_single_elec.loc[~HPC_df_single_elec.ts.isin(sig_windows) & (HPC_df_single_elec.predictor=='zrpe:phit')].groupby('unique_label').mean(numeric_only=True).reset_index()
# rank order the electrodes by their sig_OFC_z.z_beta within subj
sig_dlPFC_z['rank'] = sig_dlPFC_z.groupby('subj')['z_beta'].rank(ascending=False)

sig_dlPFC_z['overall_rank'] = sig_dlPFC_z['z_beta'].rank(ascending=False)


sig_dlPFC_z[['unique_label','z_beta','rank', 'overall_rank']].to_csv(f'{load_dir}/dlPFC_biomarker_rank.csv', index=False)



In [ ]:
ACC_df_single_elec = model_dfs[(model_dfs.region=='ACC')]
ACC_df_single_elec['unique_label'] = ACC_df_single_elec.subj + '_' + ACC_df_single_elec.unique_label.astype(str)

# compute the average z-score for zrpe:phit for sig_windows in ts
sig_windows = np.concatenate([np.arange(450, 600, 50), np.arange(1100, 1250, 50), np.arange(1350, 1500, 50)])
sig_ACC_z = ACC_df_single_elec.loc[ACC_df_single_elec.ts.isin(sig_windows) & (ACC_df_single_elec.predictor=='zrpe:phit')].groupby(['unique_label', 'subj']).mean(numeric_only=True).reset_index()
# nsig_OFC_z = HPC_df_single_elec.loc[~HPC_df_single_elec.ts.isin(sig_windows) & (HPC_df_single_elec.predictor=='zrpe:phit')].groupby('unique_label').mean(numeric_only=True).reset_index()
# rank order the electrodes by their sig_OFC_z.z_beta within subj
sig_ACC_z['rank'] = sig_ACC_z.groupby('subj')['z_beta'].rank(ascending=False)

sig_ACC_z['overall_rank'] = sig_ACC_z['z_beta'].rank(ascending=False)


sig_ACC_z[['unique_label','z_beta','rank', 'overall_rank']].to_csv(f'{load_dir}/ACC_biomarker_rank.csv', index=False)



Sanity check: I can also aggregate the single-electrode regressions and see if similar results emerge using the permutation cluster sample test. This is not the primary analysis because I think single-electrode regressions lack the statistical power to identify consistent patterns that we get from the mixed-effects framework. 

In [ ]:
event = 'cue_on'
# model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_zpmcontrol.csv')   
model_dfs = pd.read_csv(f'{load_dir}/single_elec_time_resolved_perm_{band}_{event}.csv')   

for region in ['ACC']:
    T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
        model_dfs[(model_dfs.predictor=="C(memory_outcome, Treatment(reference='hits'))[T.correct_rejections]") & (model_dfs.region==region)].z_beta.values.reshape(-1, model_dfs.ts.unique().shape[0]),
        n_permutations=1000,
        tail=0,  # Two-tailed test
    )
    print(region)
    print([p if len(c[0])>=2 else np.nan for p, c in zip(cluster_p_values, clusters)])

In [ ]:
cluster_p_values

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
sns.barplot(x=model_dfs[(model_dfs.predictor=='phit') & (model_dfs.region=='OFC')]['ts'],
            y=model_dfs[(model_dfs.predictor=='phit') & (model_dfs.region=='OFC')]['z_beta'],
            width=2, color=ofc_c, alpha=0.3, edgecolor='black', linewidth=1, errorbar='se')
ax.set(xticklabels=[], xlim=[-2, 21])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
sns.barplot(x=model_dfs[(model_dfs.predictor=='zrpe:phit') & (model_dfs.region=='HPC')]['ts'],
            y=model_dfs[(model_dfs.predictor=='zrpe:phit') & (model_dfs.region=='HPC')]['z_beta'],
            width=2, color=hpc_c, alpha=0.3, edgecolor='black', linewidth=1, errorbar='se')
ax.set(xticklabels=[], xlim=[-2, 31])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
sns.barplot(x=model_dfs[(model_dfs.predictor=='zrpe:phit') & (model_dfs.region=='dmPFC')]['ts'],
            y=model_dfs[(model_dfs.predictor=='zrpe:phit') & (model_dfs.region=='dmPFC')]['z_beta'],
            width=2, color=dmpfc_c, alpha=0.3, edgecolor='black', linewidth=1, errorbar='se')
ax.set(xticklabels=[], xlim=[-2, 31])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
sns.barplot(x=model_dfs[(model_dfs.predictor=='zpm:phit') & (model_dfs.region=='dmPFC')]['ts'],
            y=model_dfs[(model_dfs.predictor=='zpm:phit') & (model_dfs.region=='dmPFC')]['z_beta'],
            width=2, color=dlpfc_c, alpha=0.3, edgecolor='black', linewidth=1, errorbar='se')
ax.set(xticklabels=[], xlim=[-2, 31])

# Older stuff: 

The following function provides a parallelized function for extracting time-resolved data from the TFRs (averaged across band). 

In [ ]:
analysis_evs = ['feedback_start']

In [ ]:
%%time
progress_bar = tqdm(subj_ids, ascii=True, desc='Saving TFR dataframes for time-resolved data')

power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids[0:1]}

for subj_id in progress_bar:
    for day in ['Day1']:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
        for event in analysis_evs:

            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            if not os.path.exists(f'{filepath}/{event}/plots'):
                os.makedirs(f'{filepath}/{event}/plots')
            if not os.path.exists(f'{filepath}/{event}/dfs'):
                os.makedirs(f'{filepath}/{event}/dfs')

            power_epochs[subj_id][event] = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]

            epochs_to_analyze = learn_df[(learn_df.participant==subj_id)].trials.values - 1

            power_epochs[subj_id][event] = power_epochs[subj_id][event][epochs_to_analyze.astype(int)]

            power_epochs[subj_id][event].metadata = learn_df[(learn_df.participant==subj_id)]
            
#             # If you want to crop to a specific time, do so here: 
            
            
#             bands=['hfa']
            
#             # Run the regression code in parallel
#             Parallel(n_jobs=-1, verbose=1)(delayed(statistic_utils.tfr_to_bandavg_timeseries)(chan,
#                                                                               power_epochs[subj_id][event], 
#                                                                               subj_id, 
#                                                                               elec_df, 
#                                                                               event,
#                                                                               bands) for chan in power_epochs[subj_id][event].ch_names)


In [ ]:

progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

for band in ['hfa']:

    all_rpe_channels  = [] 
        
    for subj_id in progress_bar:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
        regressors = learn_df[learn_df.participant==subj_id][['rpe', 'DPRIME']]
        all_chan_df = []

        relevant_channels = elec_df[elec_df.salman_region.isin(rois)].index.values
        
        if len(relevant_channels) == 0:
            print('no relevant channels')
            continue
            
        for channel in relevant_channels:
            label = elec_df.iloc[channel].label
            timeseries = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                 np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                 :], axis=1)
            # timeseries[timeseries>=10] = np.nan
            test = statistics_utils.time_resolved_regression_single_channel(timeseries, 
                                                  regressors,standardize=True, smooth=True, 
                                                                            win_len=50, slide_len=25) # 50, 25 for HFA
            test['channel'] = channel
            test['subj'] = subj_id
            test['label'] = label
            test['region'] = elec_df.iloc[channel].salman_region
            test['band'] = band
            all_chan_df.append(test)  

        all_chan_df = pd.concat(all_chan_df).reset_index()
        all_rpe_channels.append(all_chan_df)

    all_rpe_channels = pd.concat(all_rpe_channels)
    all_rpe_channels.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress', index=False)


Now run a version that splits the OFC data by memory

In [ ]:

progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

day = 'Day1'
rois = ['OFC', 'HPC', 'AMY', 'dlPFC', 'dmPFC']
for band in ['hfa']:
    all_rpe_channels  = [] 
        
    for subj_id in progress_bar:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')
        regressors = learn_df[learn_df.participant==subj_id][['rpe', 'DPRIME']]
        all_chan_df = []

        relevant_channels = elec_df[elec_df.salman_region.isin(rois)].label.values
        
        if len(relevant_channels) == 0:
            print('no relevant channels')
            continue
            
        for label in relevant_channels:
            
            # Split the data into recalled and non-recalled: 
            memory_df = combined_df[(combined_df.condition == 'Day1') & (combined_df.participant == subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
            rec_ix = memory_df[memory_df.hits==True].trials_gamble.values.astype(int) - 1
            
            
            timeseries_rec = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                 np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                 :][rec_ix,:], axis=1)
            
            # timeseries[timeseries>=10] = np.nan
            test1 = statistics_utils.time_resolved_regression_single_channel(timeseries_rec, 
                                                  regressors.iloc[rec_ix],standardize=True, smooth=True, 
                                                                            win_len=50, slide_len=25) # 50, 25 for HFA
            test1['channel'] = channel
            test1['subj'] = subj_id
            test1['label'] = label
            test1['region'] = elec_df.loc[elec_df.label==label, 'salman_region'].values[0]
            test1['band'] = band
            test1['hit'] = True
            
            all_chan_df.append(test1)  
            
            nrec_ix = memory_df[memory_df.hits==False].trials_gamble.values.astype(int) - 1
            
            timeseries_nrec = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                 np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                 :][nrec_ix,:], axis=1)
            
            test2 = statistics_utils.time_resolved_regression_single_channel(timeseries_nrec, 
                                                  regressors.iloc[nrec_ix],standardize=True, smooth=True, 
                                                                            win_len=50, slide_len=25) # 50, 25 for HFA
            test2['channel'] = channel
            test2['subj'] = subj_id
            test2['label'] = label
            test2['region'] = elec_df.loc[elec_df.label==label, 'salman_region'].values[0]
            test2['band'] = band
            test2['hit'] = False
            
            all_chan_df.append(test2)  

        all_chan_df = pd.concat(all_chan_df).reset_index()
        all_rpe_channels.append(all_chan_df)

    all_rpe_channels = pd.concat(all_rpe_channels)
    all_rpe_channels.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split', index=False)


Now let's run a version of the time-resolved regression which runs a mixed-effects regression across all channels in a given region

Update 12/1/2024: This reveals that OFC has a nice interaction effect in the early and late time windows 

and ALSO that HPC has some notable, theta-rhythmic decreases in the back half

In [ ]:
def smooth_timeseries(timeseries, win_len, slide_len):
    slices = np.lib.stride_tricks.sliding_window_view(np.arange(timeseries.shape[-1]), win_len)[::slide_len]
    midpoints = np.ceil(np.mean(slices, axis=1))
    # Smooth the timeseries (easier to do here than to store smoothed data)
    sig = np.zeros([timeseries.shape[0], slices.shape[0]])
    for stride in range(slices.shape[0]):
        sig[:, stride] = np.nanmean(timeseries[:, slices[stride]], axis=1)
    
    return sig, midpoints

In [ ]:
# smooth criteria: 


progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'high_theta':[4, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

day = 'Day1'
rois = ['HPC']
# , 'HPC', 'AMY', 'dlPFC', 'dmPFC'
win_size = 50
slide_size = 25
for region in rois:
    for band in ['hfa']:
        all_rec_df = []
        all_nrec_df = []
            
        for subj_id in progress_bar:
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')
            regressors = learn_df[learn_df.participant==subj_id][['rpe', 'DPRIME']]
            all_chan_df = []

            relevant_channels = elec_df[elec_df.salman_region==region].label.values

            if len(relevant_channels) == 0:
                print('no relevant channels')
                continue

            # Split the data into recalled and non-recalled: 
            memory_df = combined_df[(combined_df.condition == 'Day1') & (combined_df.participant == subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
            rec_ix = memory_df[memory_df.hits==True].trials_gamble.values.astype(int) - 1
            nrec_ix = memory_df[memory_df.hits==False].trials_gamble.values.astype(int) - 1
            
            for label in relevant_channels:
                timeseries_rec = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                    np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                    :][rec_ix,:], axis=1)
                timeseries_nrec = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                    np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                    :][nrec_ix,:], axis=1)
                
                smooth_rec, midpoints = smooth_timeseries(timeseries_rec, win_size, slide_size)
                smooth_nrec, _ = smooth_timeseries(timeseries_nrec, win_size, slide_size)

                rec_dfs = []
                for rec_trials in range(len(rec_ix)):   
                    rec_df = pd.DataFrame()
                    rec_df['ts'] = midpoints * (1000/tfr_data.info['sfreq'])
                    rec_df['rec_trial'] = rec_ix[rec_trials]
                    rec_df['smoothed_data'] = smooth_rec[rec_trials, :]
                    rec_df['rpe'] = regressors.iloc[rec_ix[rec_trials]].rpe
                    rec_df['pm'] = regressors.iloc[rec_ix[rec_trials]].DPRIME
                    rec_df['hit'] = 1
                    rec_dfs.append(rec_df)

                rec_dfs = pd.concat(rec_dfs)
                rec_dfs['subj'] = subj_id
                rec_dfs['label'] = label
                rec_dfs['region'] = region
                rec_dfs['band'] = band

                all_rec_df.append(rec_dfs)

                nrec_dfs = []
                for nrec_trials in range(len(nrec_ix)):   
                    nrec_df = pd.DataFrame()
                    nrec_df['ts'] = midpoints * (1000/tfr_data.info['sfreq'])
                    nrec_df['nrec_trial'] = nrec_ix[nrec_trials]
                    nrec_df['smoothed_data'] = smooth_nrec[nrec_trials, :]
                    nrec_df['rpe'] = regressors.iloc[nrec_ix[nrec_trials]].rpe
                    nrec_df['pm'] = regressors.iloc[nrec_ix[nrec_trials]].DPRIME
                    nrec_df['hit'] = -1
                    nrec_dfs.append(nrec_df)

                nrec_dfs = pd.concat(nrec_dfs)
                nrec_dfs['subj'] = subj_id
                nrec_dfs['label'] = label
                nrec_dfs['region'] = region
                nrec_dfs['band'] = band

                all_nrec_df.append(nrec_dfs)

        all_nrec_df = pd.concat(all_nrec_df)
        all_rec_df = pd.concat(all_rec_df)

        all_nrec_df.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split_nrec_{region}', index=False)
        all_rec_df.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split_rec_{region}', index=False)
                    
                


In [ ]:
band = 'hfa' 
region = 'AMY'
all_nrec_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split_nrec_{region}')
all_rec_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split_rec_{region}')

In [ ]:
all_rec_df.reset_index(drop=True, inplace=True)

In [ ]:
all_nrec_df.reset_index(drop=True, inplace=True)

In [ ]:
all_rec_df['unique_label'] = all_rec_df['subj'] + '_' + all_rec_df['label']
all_nrec_df['unique_label'] = all_nrec_df['subj'] + '_' + all_nrec_df['label']

In [ ]:
all_betas = []
all_ps = []
all_hci = []
all_lci = []
all_df = pd.concat([all_rec_df, all_nrec_df])[['smoothed_data', 'ts', 'rpe', 'pm', 'hit', 'subj', 'unique_label']]
elec_random_dfs = []
for ts in all_df.ts.unique():
    model_df = all_df[all_df.ts==ts].copy()[['smoothed_data', 'rpe', 'hit', 'unique_label']]
    model_df.dropna(inplace=True)
    model_res = smf.mixedlm('smoothed_data ~ 1 + rpe + hit + rpe:hit', 
                re_formula='1 + rpe + hit + rpe:hit',
                data=model_df, groups=model_df['unique_label']).fit()
    
    if (model_res.summary().tables[0][3][4] == 'No') | np.isnan(model_res.pvalues[3]):
        # refit the model with a simpler random-effects approach:
        model_res = smf.mixedlm('smoothed_data ~ 1 + rpe + hit + rpe:hit', 
                    re_formula='1 + rpe + hit',
                    data=model_df, groups=model_df['unique_label']).fit()
    
    all_betas.append(model_res.params[2])
    all_lci.append(model_res.conf_int().loc['hit', 0])
    all_hci.append(model_res.conf_int().loc['hit', 1])
    all_ps.append(model_res.pvalues[2])
    if np.isnan(model_res.pvalues[2]):
        break
    elec_random_df = pd.DataFrame.from_dict(model_res.random_effects).T.reset_index().melt(id_vars='index', value_vars=['rpe']).rename(columns={'index':'unique_label', 'value':'rpe'})
    elec_random_df['ts'] = ts
    elec_random_dfs.append(elec_random_df)
elec_random_dfs = pd.concat(elec_random_dfs)

In [ ]:
# find the ts that has the highest peak rpe_hit_intx for each unique_label
elec_random_dfs.reset_index(drop=True, inplace=True)

In [ ]:
elec_random_dfs.iloc[elec_random_dfs.groupby('unique_label').idxmax().reset_index()['rpe_hit_intx'].values]

In [ ]:
sns.pointplot(data=elec_random_dfs.iloc[elec_random_dfs.groupby('unique_label').idxmax().reset_index()['rpe_hit_intx'].values], 
            x='ts', y='rpe_hit_intx')

In [ ]:
# FDR correct these p-values
from statsmodels.stats.multitest import multipletests
all_ps_corr = multipletests(all_ps, method='fdr_bh')[1]

In [ ]:
all_ps_corr

In [ ]:
all_betas

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plt.plot(all_df.ts.unique(), all_betas, color='purple')
# Instead of fill between, we can use error bars
low_err = [a-b for a, b in zip(all_betas, all_lci)]
high_err = [a-b for a, b in zip(all_hci, all_betas)]
plt.errorbar(all_df.ts.unique(), all_betas, yerr=[low_err, high_err], fmt='.', color = 'purple', markersize=17)

# plt.fill_between(all_df.ts.unique(), all_lci, all_hci, alpha=0.3)

for ts, p in zip(all_df.ts.unique(), all_ps_corr):
    if p<=0.05:
        plt.hlines(0.055, ts-25, ts+25, 'red')

plt.ylim(-0.026, 0.056)
plt.hlines(0, 0, 1500, 'black')
plt.xlabel('time (ms)')
plt.ylabel('hit (beta)')
plt.title('AMY hfa')
sns.despine()

plt.savefig(f'{plot_dir}/AMY_hfa_hit_interaction.pdf', dpi=300)

In [ ]:
rec_betas = []
rec_ps = []
rec_hci = []
rec_lci = []
for ts in all_rec_df.ts.unique():
    model_df = all_rec_df[all_rec_df.ts==ts].copy()
    model_df.dropna(inplace=True)
    model_res = smf.mixedlm('smoothed_data ~ 1 + rpe', 
                re_formula='1',
                data=model_df, groups=model_df['unique_label']).fit()
    # if model_res.pvalues[1]<=0.05:
    #     print(ts)
    #     print(model_res.summary())
    rec_betas.append(model_res.params[1])
    rec_lci.append(model_res.conf_int().loc['rpe', 0])
    rec_hci.append(model_res.conf_int().loc['rpe', 1])
    rec_ps.append(model_res.pvalues[1])


nrec_betas = []
nrec_ps = []
nrec_hci = []
nrec_lci = []
for ts in all_nrec_df.ts.unique():
    model_df = all_nrec_df[all_nrec_df.ts==ts].copy()
    model_df.dropna(inplace=True)
    model_res = smf.mixedlm('smoothed_data ~ 1 + rpe', 
                re_formula='1',
                data=model_df, groups=model_df['unique_label']).fit()
    nrec_betas.append(model_res.params[1])
    nrec_lci.append(model_res.conf_int().loc['rpe', 0])
    nrec_hci.append(model_res.conf_int().loc['rpe', 1])
    nrec_ps.append(model_res.pvalues[1])


In [ ]:
# plot the rec_betas and nrec_betas with the confidence intervals in the _lci and _hci variables
plt.plot(all_rec_df.ts.unique(), rec_betas, label='Recalled')
# plt.fill_between(all_rec_df.ts.unique(), rec_lci, rec_hci, alpha=0.3)
plt.plot(all_nrec_df.ts.unique(), nrec_betas, label='Not Recalled')
# plt.fill_between(all_nrec_df.ts.unique(), nrec_lci, nrec_hci, alpha=0.3)

# add a horizontal bar spanning the points where rec_ps is less than 0.05
for ts, p in zip(all_rec_df.ts.unique(), rec_ps):
    if p<=0.05:
        plt.hlines(0.08, ts-25, ts+25, 'blue')
for ts, p in zip(all_rec_df.ts.unique(), nrec_ps):
    if p<=0.05:
        plt.hlines(0.07, ts-25, ts+25, 'orange')


In [ ]:
rec_ps

In [ ]:
-0.031 - 0.138

In [ ]:
rec_ix

In [ ]:
band = 'hfa' 
event = 'feedback_start'                    
all_rpe_channels = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split')
                        
                        

In [ ]:
all_rpe_channels.region.unique()

In [ ]:
band = 'hfa' 
event = 'feedback_start'
all_rpe_channels = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split_OFC')
                        
          
hit_df = all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region']) & (all_rpe_channels['hit'] == True)]
miss_df = all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'])& (all_rpe_channels['hit'] == False)]
contrast_df = hit_df.copy() 
contrast_df['Original_Estimate'] = hit_df.Original_Estimate.values - miss_df.Original_Estimate.values
first_half_df = contrast_df[contrast_df.ts<=750].groupby(['subj', 'label']).mean(numeric_only=True).reset_index()
second_half_df = contrast_df[contrast_df.ts>750].groupby(['subj', 'label']).mean(numeric_only=True).reset_index()
half_quotient = first_half_df.Original_Estimate - second_half_df.Original_Estimate

In [ ]:
half_q_df = first_half_df.copy()
half_q_df['half_q'] = half_quotient
half_q_df = half_q_df[['subj', 'label', 'half_q']]
half_q_df.rename(columns={'subj':'participant'}, inplace=True)


In [ ]:
half_q_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/half_q_df', index=False)

In [ ]:
all_rpe_channels

In [ ]:
half_quotient

In [ ]:
np.nanmean(half_quotient[half_quotient<=0])

In [ ]:
sns.regplot(x=first_half_df.Original_Estimate, 
            y=second_half_df.Original_Estimate)
            

In [ ]:
sns.distplot(half_quotient)

In [ ]:
band = 'hfa'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')




In [ ]:
sns.regplot(x=half_quotient, 
            y=gamble_model_df[gamble_model_df.region=='OFC']['phit:zrpe'])

So there is no preference for early vs. late responding. 

11/30/2024: Try to carefully characterize early vs. late responders based on two criteria: 

1. must have positive, mean HFA ~ rpe weights in the either window 

2. This must be greater than the weight for misses 

In [ ]:
early_df = all_rpe_channels[(all_rpe_channels.ts>=300) & (all_rpe_channels.ts<600)].groupby(['unique_id', 'label', 'hit']).mean(numeric_only=True).reset_index()

late_df = all_rpe_channels[(all_rpe_channels.ts>=1200)].groupby(['unique_id', 'label', 'hit']).mean(numeric_only=True).reset_index()





In [ ]:
# Split the data based on hit
true_hits = early_df[early_df["hit"]]
false_hits = early_df[~early_df["hit"]]

# Merge the two subsets to compare Original_Estimate for each unique_id
merged = true_hits.merge(
    false_hits,
    on="unique_id",
    suffixes=("_true", "_false")
)

# Find unique_ids that satisfy both conditions
early_responders = merged[
    (merged["Original_Estimate_true"] > 0) & 
    (merged["Original_Estimate_true"] > merged["Original_Estimate_false"])
]["unique_id"]

In [ ]:
# Split the data based on hit
true_hits = late_df[late_df["hit"]]
false_hits = late_df[~late_df["hit"]]

# Merge the two subsets to compare Original_Estimate for each unique_id
merged = true_hits.merge(
    false_hits,
    on="unique_id",
    suffixes=("_true", "_false")
)

# Find unique_ids that satisfy both conditions
late_responders = merged[
    (merged["Original_Estimate_true"] > 0) & 
    (merged["Original_Estimate_true"] > merged["Original_Estimate_false"])
]["unique_id"]

In [ ]:
# Return a list of unique_ids that satisfy both conditions, and the lists of unique_ids that only satisfy early or late responders
only_early = early_responders[~early_responders.isin(late_responders)]
only_late = late_responders[~late_responders.isin(early_responders)]
both = early_responders[early_responders.isin(late_responders)]

In [ ]:
all_rpe_channels[all_rpe_channels.region=='OFC']

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8), dpi=300)
sns.despine()
# with PdfPages(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_time_resolved_rec_split_OFC.pdf') as pdf:
sns.pointplot(data=all_rpe_channels[(all_rpe_channels.unique_id.isin(both)) & (all_rpe_channels['index']=='rpe') & (all_rpe_channels['region']) & (all_rpe_channels['hit'] == True)],
        x='ts', y='Original_Estimate', errorbar='se', color='g', ax=ax)
sns.pointplot(data=all_rpe_channels[(all_rpe_channels.unique_id.isin(both)) & (all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'])& (all_rpe_channels['hit'] == False)],
        x='ts', y='Original_Estimate', errorbar='se', color='r', ax=ax)
# ax.set(ylim=[-0.08, 0.08])
    # pdf.savefig()
    

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8), dpi=300)
sns.despine()
with PdfPages(f'{plot_dir}/encoding_{band}_rpe_time_resolved_rec_split_OFC.pdf') as pdf:
    sns.pointplot(data=subsampled_df[(subsampled_df['index']=='rpe') & (all_rpe_channels['hit'] == True)],
         x='ts', y='Original_Estimate', errorbar='se', color='g', ax=ax)
    sns.pointplot(data=subsampled_df[(subsampled_df['index']=='rpe') & (all_rpe_channels['hit'] == False)],
         x='ts', y='Original_Estimate', errorbar='se', color='r', ax=ax)
    # ax.set(ylim=[-0.08, 0.08])
    pdf.savefig()
    

In [ ]:
all_rpe_channels['unique_elec_id'] = all_rpe_channels['subj'] + all_rpe_channels['label']

In [ ]:

above_median_subj = all_rpe_channels[all_rpe_channels.region==region].groupby(['subj', 'unique_elec_id']).mean(numeric_only=True).reset_index().subj.value_counts()[all_rpe_channels[all_rpe_channels.region==region].groupby(['subj', 'unique_elec_id']).mean(numeric_only=True).reset_index().subj.value_counts()>=median_channel_count].index.values
below_median_subj = all_rpe_channels[all_rpe_channels.region==region].groupby(['subj', 'unique_elec_id']).mean(numeric_only=True).reset_index().subj.value_counts()[all_rpe_channels[all_rpe_channels.region==region].groupby(['subj', 'unique_elec_id']).mean(numeric_only=True).reset_index().subj.value_counts()<median_channel_count].index.values

Note 11/30/2024 (12:05 PM): I think what I need to do is compute a mixed-effects regression within participant (thus accounting for each participant's electrode counts), and then pull the resulting fixed effects

In [ ]:
# Let's perform an mne cluster test between the hit = True and hit = False data: 
from mne.stats import permutation_cluster_test

# First, subsample the data so that every participant contributes the same number of electrodes per region
region = 'dlPFC'
median_channel_count = np.median(all_rpe_channels[all_rpe_channels.region==region].groupby(['subj', 'unique_elec_id']).mean(numeric_only=True).reset_index().subj.value_counts())
above_median_subj = all_rpe_channels[all_rpe_channels.region==region].groupby(['subj', 'unique_elec_id']).mean(numeric_only=True).reset_index().subj.value_counts()[all_rpe_channels[all_rpe_channels.region==region].groupby(['subj', 'unique_elec_id']).mean(numeric_only=True).reset_index().subj.value_counts()>=median_channel_count].index.values
below_median_subj = all_rpe_channels[all_rpe_channels.region==region].groupby(['subj', 'unique_elec_id']).mean(numeric_only=True).reset_index().subj.value_counts()[all_rpe_channels[all_rpe_channels.region==region].groupby(['subj', 'unique_elec_id']).mean(numeric_only=True).reset_index().subj.value_counts()<median_channel_count].index.values

# min_elecs_per_participant = all_rpe_channels[all_rpe_channels.region==region].groupby(['subj', 'unique_elec_id']).mean(numeric_only=True).reset_index().subj.value_counts().min()
# sub-sample the data to have the same number of electrodes per participant using min_elecs_per_participant
# Create an empty list to store subsampled groups
# subsampled_data = []

# # Iterate through unique subjects
# for subj in all_rpe_channels['subj'].unique():
#     # Filter rows for the current subject
#     subj_data = all_rpe_channels[all_rpe_channels['subj'] == subj]
    
#     # Identify unique electrodes for the subject
#     unique_elec_ids = subj_data['unique_elec_id'].unique()
    
#     # Subsample up to min_elecs_per_participant unique electrodes
#     sampled_elec_ids = pd.Series(unique_elec_ids).sample(n=min(min_elecs_per_participant, len(unique_elec_ids)), random_state=42).tolist()
    
#     # Filter the dataframe for the selected unique_elec_ids
#     subsampled_group = subj_data[subj_data['unique_elec_id'].isin(sampled_elec_ids)]
    
#     # Append to the list
#     subsampled_data.append(subsampled_group)

# # Concatenate all the subsampled groups
# subsampled_df = pd.concat(subsampled_data, ignore_index=True)

subsampled_df = all_rpe_channels[(all_rpe_channels.region==region) & (all_rpe_channels.subj.isin(below_median_subj))]
# second, reshape the data into a 2D array according to 'ts'
hit_data = subsampled_df[(subsampled_df['index']=='rpe') & (subsampled_df['hit'] == True)].Original_Estimate.values.reshape(-1, 29)
miss_data = subsampled_df[(subsampled_df['index']=='rpe') & (subsampled_df['hit'] == False)].Original_Estimate.values.reshape(-1, 29)

T_obs, clusters, cluster_p_values, H0 = permutation_cluster_test([hit_data, miss_data], n_permutations=1000)


In [ ]:
cluster_p_values

In [ ]:
# # Assuming your dataframe is called 'df'
# all_rpe_channels['any_sig'] = 0
# all_rpe_channels['any_sig'] = (all_rpe_channels.groupby(['subj', 'label'])['P_Value']
#                    .transform(lambda x: np.any(x <= (0.05/27)))
#                    .astype(int))
# # all_rpe_channels['best_ts'] = all_rpe_channels.groupby(['subj', 'label']).apply(
# #     lambda x: x.loc[x['P_Value'].idxmin(), 'ts']
# # ).reset_index(level=[0,1], drop=True)

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=300)
# sns.despine()
# sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'].isin(['dmPFC']))],
#      x='ts', y='Original_Estimate', errorbar='se', hue='any_sig', ax=ax)


In [ ]:
# # plt.hist(all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels.region.isin(['AMY']))].best_ts)
# # plt.hist(all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels.region.isin(['HPC']))].best_ts)
# fig, ax = plt.subplots(1, 1, figsize=(5, 4), dpi=300)

# sns.histplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels.region.isin(['OFC', 'AINS']))].reset_index(), x='best_ts', hue='region', 
#          element='step', stat="density", common_norm=False, ax=ax)
    

In [ ]:
band = 'theta'
all_rpe_channels = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress')


In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
# sns.despine()
# with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_Intercept_time_resolved.pdf') as pdf:
#     sns.lineplot(data=all_rpe_channels[(all_rpe_channels['index']=='Intercept')],
#          x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
#     ax.set(ylim=[-0.1, 0.1])
#     pdf.savefig()
    

In [ ]:
# Load results of single-electrode regression
band = 'theta'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_rpe_pm_phit_eleclevel_df')


gamble_model_df[gamble_model_df.rpe<-1.96][gamble_model_df.region=='dmPFC']

In [ ]:
filtered_df = all_rpe_channels[(all_rpe_channels['index']=='rpe')]
filtered_df = filtered_df[filtered_df['subj'].isin(gamble_model_df[gamble_model_df.rpe<-1.96][gamble_model_df.region=='dmPFC'].participant
                                                  ) & filtered_df['label'].isin(gamble_model_df[gamble_model_df.rpe<-1.96][gamble_model_df.region=='dmPFC'].channel)]

fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
sns.despine()
with PdfPages(f'{plot_dir}/encoding_{band}_rpe_time_resolved_sigdmpfc.pdf') as pdf:

    sns.pointplot(data=filtered_df,
         x='ts', y='Original_Estimate', errorbar='se', ax=ax, color='red')
    
    pdf.savefig()

In [ ]:
filtered_df = all_rpe_channels[(all_rpe_channels['index']=='rpe')]
filtered_df = filtered_df[filtered_df['subj'].isin(gamble_model_df[gamble_model_df.rpe<-1.96][gamble_model_df.region=='OFC'].participant
                                                  ) & filtered_df['label'].isin(gamble_model_df[gamble_model_df.rpe<-1.96][gamble_model_df.region=='OFC'].channel)]

fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
sns.despine()
with PdfPages(f'{plot_dir}/encoding_{band}_rpe_time_resolved_sigofc.pdf') as pdf:

    sns.pointplot(data=filtered_df,
         x='ts', y='Original_Estimate', errorbar='se', ax=ax, color='yellow')
    
    pdf.savefig()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
sns.despine()
with PdfPages(f'{plot_dir}/encoding_{band}_rpe_time_resolved.pdf') as pdf:
    sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'].isin(['AMY','OFC']))],
         x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
    # ax.set(ylim=[-0.075, 0.075])
    pdf.savefig()
    

In [ ]:
band = 'hfa'
all_rpe_channels = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8), dpi=300)
sns.despine()
with PdfPages(f'{plot_dir}/encoding_{band}_rpe_time_resolved.pdf') as pdf:
    sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'])],
         x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
    ax.set(ylim=[-0.05, 0.05])
    pdf.savefig()
    

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
# sns.despine()
# with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_time_resolved.pdf') as pdf:
#     sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'].isin(['HPC','dmPFC']))],
#          x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
#     ax.set(ylim=[-0.1, 0.1])
#     pdf.savefig()
    

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
# sns.despine()
# # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_hfa_rpe_eleclevel_time_resolved.pdf') as pdf:
# sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels.region.isin(['AINS']))],
#          x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
#     # pdf.savefig()

In [ ]:
df_max = all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'].isin(['AMY','OFC']))].groupby(['subj', 'label', 'region']).apply(
    lambda x: x.loc[x['P_Value'].idxmin(), 'ts']).reset_index().rename(columns={0:'peak_ts'})

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=300)
sns.despine()
with PdfPages(f'{plot_dir}/encoding_hfa_rpe_eleclevel_max_ts.pdf') as pdf:
    sns.histplot(data=df_max, 
                 x='peak_ts', hue='region', 
             element='step', stat="density", common_norm=False, ax=ax)
    pdf.savefig()
    


In [ ]:
ttest_ind(df_max[df_max.region=='AMY'].peak_ts,
          df_max[df_max.region=='OFC'].peak_ts)

In [ ]:
all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'].isin(['AMY','OFC']))]




In [ ]:

progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

for band in ['hfa']:
    all_rpe_channels  = [] 
        
    for subj_id in progress_bar:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
        regressors = learn_df[learn_df.participant==subj_id][['rpe', 'DPRIME']]
        all_chan_df = []

        relevant_channels = elec_df[elec_df.salman_region.isin(['AMY', 'HPC', 'OFC', 'dmPFC'])].index.values
        
        for channel in relevant_channels:
            label = elec_df.iloc[channel].label
            timeseries = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                 np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                 :], axis=1)

            test = statistics_utils.time_resolved_regression_single_channel(timeseries, 
                                                  regressors,standardize=True, smooth=True, 
                                                                            win_len=50, slide_len=25) # 50, 25 for HFA
            test['channel'] = channel
            test['subj'] = subj_id
            test['label'] = label
            test['region'] = elec_df.iloc[channel].salman_region
            test['band'] = band
            all_chan_df.append(test)  

        all_chan_df = pd.concat(all_chan_df).reset_index()
        all_rpe_channels.append(all_chan_df)

    all_rpe_channels = pd.concat(all_rpe_channels)
    all_rpe_channels.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rpe_splitphit', index=False)


In [ ]:
#  %%time
# progress_bar = tqdm(subj_ids, ascii=True, desc='Saving TFR dataframes for time-resolved data')
# freqs = np.logspace(*np.log10([2, 200]), num=30)
# event = 'feedback_start'
# all_phit_channels  = [] 
# for subj_id in progress_bar:
#     elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
    
#     filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
#     tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
#     all_hfa = np.nanmean(tfr_data.data[:, :, np.where(freqs>70)[0], :], axis=2)
                
#     memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
#     memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
#     memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
#     epochs_to_include = memory_df.trials.values.astype(int) - 1
    
#     regressors = memory_df[['phit']]
                
#     all_chan_df = []
    
#     for channel in range(all_hfa.shape[1]):
#         test = statistics_utils.time_resolved_regression(all_hfa[epochs_to_include, channel, :], 
#                                               regressors, win_len=100, 
#                              slide_len=25, 
#                              standardize=True, 
#                              sr=tfr_data.info['sfreq'])
#         test['channel'] = channel
#         test['subj'] = subj_id
#         test['label'] = elec_df.iloc[channel].label
#         test['region'] = elec_df.iloc[channel].salman_region
#         all_chan_df.append(test)  

#     all_chan_df = pd.concat(all_chan_df).reset_index()
#     relevant_channels = elec_df[elec_df.salman_region.isin(['HPC'])].index.values
#     relevant_channels_df = all_chan_df[all_chan_df.channel.isin(relevant_channels)]
#     all_phit_channels.append(relevant_channels_df)

# all_phit_channels = pd.concat(all_phit_channels)


In [ ]:

progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

for band in ['hfa']:
    all_phit_channels  = [] 
        
    for subj_id in progress_bar:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]

        memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
        memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
        memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
        epochs_to_include = memory_df.trials.values.astype(int) - 1

        regressors = memory_df[['phit']]

        all_chan_df = []

        relevant_channels = elec_df[elec_df.salman_region.isin(['AMY', 'HPC', 'OFC', 'dmPFC'])].index.values
        
        for channel in relevant_channels:
            label = elec_df.iloc[channel].label
            timeseries = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                 np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                 :], axis=1)
            timeseries = timeseries[epochs_to_include]
            test = statistics_utils.time_resolved_regression_single_channel(timeseries, 
                                                  regressors,standardize=True, smooth=True, 
                                                                            win_len=50, slide_len=25) # 50, 25 for HFA
            test['channel'] = channel
            test['subj'] = subj_id
            test['label'] = label
            test['region'] = elec_df.iloc[channel].salman_region
            test['band'] = band
            all_chan_df.append(test)  

        all_chan_df = pd.concat(all_chan_df).reset_index()
        all_phit_channels.append(all_chan_df)

    all_phit_channels = pd.concat(all_phit_channels)
    all_phit_channels.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_phit', index=False)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
sns.despine()
with PdfPages(f'{plot_dir}/encoding_{band}_phit_time_resolved.pdf') as pdf:
    sns.pointplot(data=all_phit_channels[(all_phit_channels['index']=='phit') & (all_phit_channels['region'].isin(['AMY','HPC']))],
         x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
    # ax.set(ylim=[-0.075, 0.075])
    pdf.savefig()
    

In [ ]:
test = 'llesion5-llesion6'


In [ ]:
'lesion' in test

In [ ]:
df_max = all_phit_channels.loc[all_phit_channels.groupby(['subj', 'label'])['phit'].idxmax()]


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 4), dpi=300)
sns.despine()
with PdfPages(f'{plot_dir}/encoding_hfa_phit_eleclevel_max_ts.pdf') as pdf:
    sns.histplot(data=df_max, x='ts',
             element='step', stat="density", common_norm=False, ax=ax)
    pdf.savefig()
    


In [ ]:
# Apply FDR correction to each column
from statsmodels.stats import multitest
        
# Apply FDR correction to each numeric column
def apply_fdr_correction(series):
    _, p_adjusted, _, _ = multitest.multipletests(series, method='fdr_bh')
    return p_adjusted

        
coef_dfs = []
p_dfs = [] 
for subj in glms.participant.unique():
    subj_df = glms[glms.participant==subj]
    for chan in subj_df.chan.unique():
        
        test_df = subj_df[subj_df.chan==chan].groupby(['fband', 'params']).mean().reset_index()
        # Pivot the DataFrame
        coef_df = test_df.pivot(index='fband', columns='params', values='coef').reset_index()
        # Reset the index to make 'fband' a column again
        # If you don't want to reset the index, you can skip the next line
        coef_df = coef_df.reset_index(drop=True)
        # Rename the index column
        coef_df.columns.name = None

        # Pivot the DataFrame
        pval_df = test_df.pivot(index='fband', columns='params', values='p').reset_index()
        # Reset the index to make 'fband' a column again
        # If you don't want to reset the index, you can skip the next line
        pval_df = pval_df.reset_index(drop=True)
        # Rename the index column
        pval_df.columns.name = None

        # Select only numeric columns
        numeric_columns = pval_df.select_dtypes(include='number')
        
        fdr_df = numeric_columns.apply(apply_fdr_correction)

        fdr_df['fband'] = pval_df['fband']

        coef_df['participant'] = subj
        coef_df['chan'] = chan
        coef_df['region'] = subj_df[subj_df.chan==chan].region.unique()[0]
        coef_dfs.append(coef_df)

        fdr_df['participant'] = subj
        fdr_df['chan'] = chan
        fdr_df['region'] = subj_df[subj_df.chan==chan].region.unique()[0]
        p_dfs.append(fdr_df)

In [ ]:
all_p_df= pd.concat(p_dfs)
all_p_df = all_p_df[all_p_df.region!='Unknown']
numeric_columns = all_p_df.select_dtypes(include='number')

In [ ]:
all_coef_df = pd.concat(coef_dfs)
all_coef_df = all_coef_df[all_coef_df.region!='Unknown']


In [ ]:
g = sns.FacetGrid(all_coef_df, 
                  row='region', 
                  height=7.5, sharey=True, sharex=True)

g.map(sns.barplot, "fband", "zrpe",  errorbar='se', order=['theta', 'alpha', 'beta', 'slowgamma', 'hfa'])


In [ ]:
g = sns.FacetGrid(all_coef_df, 
                  row='region', 
                  height=7.5, sharey=True, sharex=True)

g.map(sns.barplot, "fband", "zpm",  errorbar='se', order=['theta', 'alpha', 'beta', 'slowgamma', 'hfa'])


## Load the dataframes for the time-resolved regression

In [ ]:
band_regress_dfs = []
for subj_id in progress_bar:
    filepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    for day in ['Day1']:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
        for event in analysis_evs:
            for chan in elec_df.label.unique():
                if chan == 'Unknown':
                    continue
                else:
                    band_regress_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_regressed_surr.csv')
                    band_regress_df = band_regress_df[band_regress_df.region!='Unknown']
                    # add subject - oops
                    band_regress_df['subj'] = subj_id
                    # check significance
                    band_regress_df['rpe_sig_upper'] = np.nan
                    band_regress_df['rpe_sig_lower'] = np.nan
                    band_regress_df['DPRIME_sig_upper'] = np.nan
                    band_regress_df['DPRIMe_sig_lower'] = np.nan
                    for band in band_regress_df.fband.unique():
                        band_regress_df['rpe_sig_upper'][band_regress_df.fband==band] = int(any(band_regress_df[band_regress_df.fband==band].rpe_p_upper_fdr<=0.025))
                        band_regress_df['rpe_sig_lower'][band_regress_df.fband==band] = int(any(band_regress_df[band_regress_df.fband==band].rpe_p_lower_fdr<=0.025))
                        band_regress_df['DPRIME_sig_upper'][band_regress_df.fband==band] = int(any(band_regress_df[band_regress_df.fband==band].DPRIME_p_upper_fdr<=0.025))
                        band_regress_df['DPRIMe_sig_lower'][band_regress_df.fband==band] = int(any(band_regress_df[band_regress_df.fband==band].DPRIME_p_lower_fdr<=0.025))
                    band_regress_dfs.append(band_regress_df)

In [ ]:
plot_df = band_regress_dfs[104]
for band in ['theta', 'alpha', 'beta', 'hfa']:
    for feature in ['rpe', 'DPRIME']:
        sig_time_pts_upper = plot_df[(plot_df.fband==band) & (plot_df[f'{feature}_p_upper_fdr']<=0.025)].ts
        sig_time_pts_lower = plot_df[(plot_df.fband==band) & (plot_df[f'{feature}_p_lower_fdr']<=0.025)].ts
        
        if (len(sig_time_pts_upper) > 0) | (len(sig_time_pts_lower) >0):
            fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
            sns.lineplot(data=plot_df[plot_df.fband==band], x='ts', y=f'{feature}')
            plt.plot(sig_time_pts_upper,
                     [plt.ylim()[1]]*len(sig_time_pts_upper), 'o')
            plt.plot(sig_time_pts_lower,
                     [plt.ylim()[0]]*len(sig_time_pts_lower), 'o')
            plt.suptitle(f'{band}_{feature}_{plot_df.region.unique()[0]}')

In [ ]:
features = ['rpe', 'DPRIME']
sig_chans = {f'{x}':[] for x in features}
sig_regions = {f'{x}':[] for x in features}
for plot_df in band_regress_dfs:
    for band in ['beta']:
        for feature in features:
            sig_time_pts_upper = plot_df[(plot_df.fband==band) & (plot_df[f'{feature}_p_upper_fdr']<=0.025)].ts
            sig_time_pts_lower = plot_df[(plot_df.fband==band) & (plot_df[f'{feature}_p_lower_fdr']<=0.025)].ts
            if (len(sig_time_pts_upper) > 0) | (len(sig_time_pts_lower) >0):
                sig_chans[feature].append(plot_df.chan.unique()[0]) 
                sig_regions[feature].append(plot_df.region.unique()[0])

In [ ]:
from collections import Counter
sig_chan_dfs = [] 
for feature in features:
    sig_chan_df = pd.DataFrame(columns=['region', 'count', 'feature'])
    sig_chan_df['region'] = Counter(sig_regions[feature]).keys()
    sig_chan_df['count'] = Counter(sig_regions[feature]).values()
    sig_chan_df['feature'] = feature 
    sig_chan_dfs.append(sig_chan_df) 
sig_chan_dfs = pd.concat(sig_chan_dfs)  

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 4), dpi=300)
sns.barplot(data=sig_chan_dfs[sig_chan_dfs.region!='Unknown'], x='region', y='count', hue='feature')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 4), dpi=300)
sns.barplot(data=sig_chan_dfs[sig_chan_dfs.region!='Unknown'], x='region', y='count', hue='feature')
